<a href="https://colab.research.google.com/github/YaokunLin/ActuaryProgrammer/blob/main/joint_treasury_funding_collateral_optimization_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Joint funding-efficiency and collateral-allocation optimization example.

This synthetic interview-preparation example illustrates a central Treasury
optimization problem:

    How should a global trading firm meet margin and USD liquidity needs at
    minimum risk-adjusted annual cost, when the same security can either be
    posted for margin or pledged to raise cash through repo, but not both?

The model jointly chooses:

1. Margin collateral posted to each counterparty.
2. Securities pledged into repo to raise USD cash.
3. Other cash-funding routes, including an EUR-to-USD FX swap.

It enforces:

* counterparty eligibility and haircuts;
* security-inventory conservation (no double use of collateral);
* a USD cash balance for settlements, margin, and liquidity buffer;
* funding-route and repo-dealer capacities; and
* a minimum amount of stable (30+ day) external funding.

All amounts are USD millions and all costs are annualized basis points.

The data are deliberately synthetic and do not describe HRT's actual
platform, positions, counterparties, or financing arrangements.

Install:
    python -m pip install numpy pandas scipy

Run:
    python joint_treasury_funding_collateral_optimization.py
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import OptimizeResult, linprog


# ============================================================================
# DATA CLASSES
# ============================================================================


@dataclass(frozen=True)
class CashPolicy:
    """USD cash requirements that the joint optimizer must satisfy."""

    opening_cash_usd_m: float
    settlement_cash_need_usd_m: float
    minimum_liquidity_buffer_usd_m: float
    minimum_stable_funding_usd_m: float
    stable_tenor_days: int = 30

    @property
    def required_non_margin_cash_usd_m(self) -> float:
        """Cash retained for settlements and the end-of-day buffer."""
        return (
            self.settlement_cash_need_usd_m
            + self.minimum_liquidity_buffer_usd_m
        )


@dataclass
class JointSolution:
    """Solver output plus tables needed to explain the recommendation."""

    result: OptimizeResult
    variables: pd.DataFrame
    constraints: pd.DataFrame


# ============================================================================
# SYNTHETIC INPUT DATA
# ============================================================================


def make_synthetic_inputs() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    CashPolicy,
]:
    """Create a small synthetic global funding and collateral data set."""

    # ------------------------------------------------------------------------
    # Available assets
    # ------------------------------------------------------------------------

    assets = pd.DataFrame(
        {
            "asset": [
                "USD_CASH",
                "T_BILL",
                "UST_2Y",
                "UST_10Y",
                "CORP_BOND",
            ],
            "is_security": [
                False,
                True,
                True,
                True,
                True,
            ],
            "usable_inventory_usd_m": [
                np.nan,
                90.0,
                80.0,
                70.0,
                50.0,
            ],
            "opportunity_cost_bps": [
                70.0,
                8.0,
                14.0,
                25.0,
                40.0,
            ],
        }
    )

    # ------------------------------------------------------------------------
    # Margin requirements
    # ------------------------------------------------------------------------

    margin_requirements = pd.DataFrame(
        {
            "counterparty": [
                "CCP_A",
                "PB_B_CASH_ONLY",
                "CCP_C",
            ],
            "required_margin_usd_m": [
                132.6428,
                64.8476,
                97.2714,
            ],
        }
    )

    # ------------------------------------------------------------------------
    # Margin collateral terms
    # ------------------------------------------------------------------------

    margin_terms = pd.DataFrame(
        [
            ("CCP_A", "USD_CASH", 0.000, 0.2),
            ("CCP_A", "T_BILL", 0.010, 0.5),
            ("CCP_A", "UST_2Y", 0.020, 0.7),
            ("CCP_A", "UST_10Y", 0.040, 1.0),
            ("CCP_A", "CORP_BOND", 0.080, 1.5),

            ("PB_B_CASH_ONLY", "USD_CASH", 0.000, 0.1),

            ("CCP_C", "USD_CASH", 0.000, 0.2),
            ("CCP_C", "T_BILL", 0.015, 0.6),
            ("CCP_C", "UST_2Y", 0.030, 0.8),
            ("CCP_C", "UST_10Y", 0.050, 1.1),
        ],
        columns=[
            "counterparty",
            "asset",
            "haircut",
            "transfer_cost_bps",
        ],
    )

    # ------------------------------------------------------------------------
    # Repo routes
    # ------------------------------------------------------------------------

    repo_terms = pd.DataFrame(
        [
            (
                "D1_TBILL_OVERNIGHT",
                "REPO_D1",
                "T_BILL",
                0.02,
                20.0,
                50.0,
                1,
            ),
            (
                "D1_UST2_OVERNIGHT",
                "REPO_D1",
                "UST_2Y",
                0.03,
                28.0,
                45.0,
                1,
            ),
            (
                "D2_UST10_ONE_WEEK",
                "REPO_D2",
                "UST_10Y",
                0.05,
                45.0,
                40.0,
                7,
            ),
            (
                "D2_CORP_ONE_MONTH",
                "REPO_D2",
                "CORP_BOND",
                0.12,
                70.0,
                25.0,
                30,
            ),
        ],
        columns=[
            "route",
            "dealer",
            "asset",
            "haircut",
            "repo_cash_cost_bps",
            "maximum_collateral_usd_m",
            "tenor_days",
        ],
    )

    # ------------------------------------------------------------------------
    # Dealer-level repo limits
    # ------------------------------------------------------------------------

    repo_dealer_limits = pd.DataFrame(
        {
            "dealer": [
                "REPO_D1",
                "REPO_D2",
            ],
            "maximum_cash_proceeds_usd_m": [
                65.0,
                35.0,
            ],
        }
    )

    # ------------------------------------------------------------------------
    # Other cash funding sources
    # ------------------------------------------------------------------------

    cash_funding_sources = pd.DataFrame(
        [
            (
                "EUR_SURPLUS_VIA_FX_SWAP",
                "fx_swap",
                50.0,
                25.0,
                30,
            ),
            (
                "USD_PB_CASH_FACILITY",
                "pb_facility",
                85.0,
                60.0,
                1,
            ),
            (
                "USD_TERM_FACILITY",
                "term_facility",
                95.0,
                50.0,
                90,
            ),
        ],
        columns=[
            "source",
            "funding_type",
            "all_in_cost_bps",
            "maximum_cash_usd_m",
            "tenor_days",
        ],
    )

    # ------------------------------------------------------------------------
    # Liquidity policy
    # ------------------------------------------------------------------------

    cash_policy = CashPolicy(
        opening_cash_usd_m=45.0,
        settlement_cash_need_usd_m=20.0,
        minimum_liquidity_buffer_usd_m=25.0,
        minimum_stable_funding_usd_m=20.0,
        stable_tenor_days=30,
    )

    return (
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
    )


# ============================================================================
# PRINT INPUTS
# ============================================================================


def print_model_inputs(
    assets: pd.DataFrame,
    margin_requirements: pd.DataFrame,
    margin_terms: pd.DataFrame,
    repo_terms: pd.DataFrame,
    repo_dealer_limits: pd.DataFrame,
    cash_funding_sources: pd.DataFrame,
    cash_policy: CashPolicy,
) -> None:
    """Print the Treasury inputs in an interview-friendly format."""

    pd.set_option("display.width", 280)
    pd.set_option("display.max_columns", 40)
    pd.set_option(
        "display.float_format",
        lambda value: f"{value:,.4f}",
    )

    total_margin = float(
        margin_requirements["required_margin_usd_m"].sum()
    )

    total_security_inventory = float(
        assets.loc[
            assets["is_security"],
            "usable_inventory_usd_m",
        ].sum()
    )

    free_cash_available = (
        cash_policy.opening_cash_usd_m
        - cash_policy.settlement_cash_need_usd_m
        - cash_policy.minimum_liquidity_buffer_usd_m
    )

    # ========================================================================
    # 1. Problem
    # ========================================================================

    print("\n" + "=" * 110)
    print("1. PROBLEM TO SOLVE")
    print("=" * 110)

    print(
        """
We need to satisfy BOTH:

    A. Margin requirements at CCPs / prime brokers
    B. USD liquidity requirements

while minimizing annual economic cost.

The main trade-off is:

                         SECURITY INVENTORY
                                |
                --------------------------------
                |                              |
                v                              v
        POST FOR MARGIN                 PLEDGE INTO REPO
                |                              |
                v                              v
        Margin coverage                   Raise USD cash
                                                |
                                                v
                                   Settlements / liquidity

The same security cannot be used for both purposes at the same time.
"""
    )

    # ------------------------------------------------------------------------
    # Margin and inventory
    # ------------------------------------------------------------------------

    print("-" * 75)
    print("MARGIN AND SECURITY INVENTORY")
    print("-" * 75)

    print(
        f"Total margin requirement:               "
        f"${total_margin:,.2f}m"
    )

    print(
        f"Total available security inventory:     "
        f"${total_security_inventory:,.2f}m"
    )

    # ------------------------------------------------------------------------
    # Cash position
    # ------------------------------------------------------------------------

    print("\n" + "-" * 75)
    print("USD CASH POSITION")
    print("-" * 75)

    print(
        f"Opening cash                             "
        f"${cash_policy.opening_cash_usd_m:,.0f}m"
    )

    print(
        f"Needed for settlements                  "
        f"-${cash_policy.settlement_cash_need_usd_m:,.0f}m"
    )

    print(
        f"Must remain as liquidity buffer         "
        f"-${cash_policy.minimum_liquidity_buffer_usd_m:,.0f}m"
    )

    print(
        "                                         -----"
    )

    print(
        f"Free cash available                      "
        f"${free_cash_available:,.0f}m"
    )

    # ------------------------------------------------------------------------
    # Funding policy
    # ------------------------------------------------------------------------

    print("\n" + "-" * 75)
    print("FUNDING POLICY")
    print("-" * 75)

    print(
        f"Minimum stable funding "
        f"({cash_policy.stable_tenor_days}+ days):       "
        f"${cash_policy.minimum_stable_funding_usd_m:,.2f}m"
    )

    print(
        """
Interpretation:

Because free USD cash is $0m, any USD cash posted as margin generally
has to be replaced through repo, FX swaps, or another funding source.

The stable-funding requirement is different:
at least $20m of external funding must have a tenor of 30 days or more.
"""
    )

    # ========================================================================
    # 2. Assets
    # ========================================================================

    print("\n" + "=" * 110)
    print("2. AVAILABLE ASSETS")
    print("=" * 110)

    print(
        assets[
            [
                "asset",
                "is_security",
                "usable_inventory_usd_m",
                "opportunity_cost_bps",
            ]
        ].to_string(index=False)
    )

    print(
        """
Interpretation:

usable_inventory_usd_m
    Maximum market value of each security available for margin or repo.

opportunity_cost_bps
    Economic value lost when that asset becomes encumbered.

For example, if a Treasury is posted to a CCP, it can no longer be used
for repo or another collateral purpose.
"""
    )

    # ========================================================================
    # 3. Margin requirements
    # ========================================================================

    print("\n" + "=" * 110)
    print("3. COUNTERPARTY MARGIN REQUIREMENTS")
    print("=" * 110)

    print(
        margin_requirements.to_string(index=False)
    )

    # ========================================================================
    # 4. Collateral eligibility
    # ========================================================================

    print("\n" + "=" * 110)
    print("4. COLLATERAL ELIGIBILITY")
    print("=" * 110)

    eligibility = margin_terms.copy()
    eligibility["eligible"] = "YES"

    eligibility_matrix = (
        eligibility.pivot(
            index="asset",
            columns="counterparty",
            values="eligible",
        )
        .fillna("-")
        .reset_index()
    )

    print(
        eligibility_matrix.to_string(index=False)
    )

    print(
        """
A "-" means that asset is not eligible at that counterparty.

PB_B_CASH_ONLY accepts only USD cash.

That restriction matters because the optimizer must make sure enough
cash can be funded for PB_B while still preserving the liquidity buffer.
"""
    )

    # ========================================================================
    # 5. Margin economics
    # ========================================================================

    print("\n" + "=" * 110)
    print("5. MARGIN COLLATERAL ECONOMICS")
    print("=" * 110)

    margin_economics = margin_terms.merge(
        assets[
            [
                "asset",
                "opportunity_cost_bps",
            ]
        ],
        on="asset",
        how="left",
        validate="many_to_one",
    )

    margin_economics["raw_cost_bps"] = (
        margin_economics["opportunity_cost_bps"]
        + margin_economics["transfer_cost_bps"]
    )

    margin_economics["effective_margin_per_1"] = (
        1.0 - margin_economics["haircut"]
    )

    margin_economics["effective_cost_bps"] = (
        margin_economics["raw_cost_bps"]
        / margin_economics["effective_margin_per_1"]
    )

    print(
        margin_economics[
            [
                "counterparty",
                "asset",
                "haircut",
                "opportunity_cost_bps",
                "transfer_cost_bps",
                "raw_cost_bps",
                "effective_margin_per_1",
                "effective_cost_bps",
            ]
        ].to_string(index=False)
    )

    print(
        """
Margin collateral cost:

    raw cost
        = opportunity cost
        + transfer cost

Haircut determines usable margin:

    effective margin
        = collateral market value x (1 - haircut)

Therefore:

                          raw cost
    effective cost = ------------------
                        1 - haircut


Example: UST_10Y at CCP_A

    Opportunity cost = 25 bps
    Transfer cost    =  1 bps
                         ------
    Raw cost         = 26 bps

    Haircut = 4%

    Effective cost
        = 26 / 0.96
        = 27.08 bps
"""
    )

    # ========================================================================
    # 6. Repo routes
    # ========================================================================

    print("\n" + "=" * 110)
    print("6. REPO FUNDING OPTIONS")
    print("=" * 110)

    repo_economics = repo_terms.merge(
        assets[
            [
                "asset",
                "opportunity_cost_bps",
            ]
        ],
        on="asset",
        how="left",
        validate="many_to_one",
    )

    repo_economics["cash_per_1_collateral"] = (
        1.0 - repo_economics["haircut"]
    )

    repo_economics[
        "cost_per_collateral_bps"
    ] = (
        repo_economics["opportunity_cost_bps"]
        + repo_economics["repo_cash_cost_bps"]
        * repo_economics["cash_per_1_collateral"]
    )

    repo_economics[
        "effective_cash_cost_bps"
    ] = (
        repo_economics["cost_per_collateral_bps"]
        / repo_economics["cash_per_1_collateral"]
    )

    print(
        repo_economics[
            [
                "route",
                "dealer",
                "asset",
                "tenor_days",
                "haircut",
                "repo_cash_cost_bps",
                "opportunity_cost_bps",
                "maximum_collateral_usd_m",
                "cash_per_1_collateral",
                "effective_cash_cost_bps",
            ]
        ].to_string(index=False)
    )

    print(
        """
Repo economics:

    $1 security pledged
        -> haircut
        -> less than $1 cash received

Economic cost includes:

    1. opportunity cost of using the security
    2. repo financing cost on the cash raised

Therefore the optimizer should not simply choose the lowest quoted
repo rate.
"""
    )

    # ========================================================================
    # 7. Dealer capacity
    # ========================================================================

    print("\n" + "=" * 110)
    print("7. REPO DEALER CAPACITY")
    print("=" * 110)

    print(
        repo_dealer_limits.to_string(index=False)
    )

    print(
        """
These limits can represent:

    - credit capacity
    - concentration limits
    - dealer balance-sheet capacity
    - operational capacity
    - internal Treasury limits
"""
    )

    # ========================================================================
    # 8. Other funding
    # ========================================================================

    print("\n" + "=" * 110)
    print("8. OTHER USD FUNDING SOURCES")
    print("=" * 110)

    print(
        cash_funding_sources.to_string(index=False)
    )

    print(
        """
Example:

EUR_SURPLUS_VIA_FX_SWAP

means Treasury has surplus EUR liquidity and converts it into USD funding
through an FX swap.

The 50 bps is modeled as an all-in annualized USD funding cost.
"""
    )

    # ========================================================================
    # 9. LP formulation
    # ========================================================================

    print("\n" + "=" * 110)
    print("9. LINEAR PROGRAM FORMULATION")
    print("=" * 110)

    print(
        """
DECISION VARIABLES
------------------

M[c,a]
    Market value of asset a posted to counterparty c.

R[r]
    Market value of security pledged through repo route r.

F[f]
    USD cash raised using another funding source.


OBJECTIVE
---------

Minimize:

      collateral economic cost
    + repo funding economic cost
    + FX / facility funding cost


CONSTRAINT 1: MARGIN COVERAGE
-----------------------------

For each counterparty:

    SUM(
        collateral market value
        x
        (1 - haircut)
    )

        >= required margin


CONSTRAINT 2: SECURITY INVENTORY
--------------------------------

For each security:

    margin use
    +
    repo use

        <= available inventory


CONSTRAINT 3: USD CASH BALANCE
------------------------------

    opening cash
    + repo proceeds
    + other funding
    - cash posted as margin
    - settlement cash

        >= minimum liquidity buffer


CONSTRAINT 4: STABLE FUNDING
----------------------------

    funding with tenor >= 30 days

        >= minimum stable funding


CONSTRAINT 5: REPO DEALER CAPACITY
----------------------------------

    repo cash raised from dealer

        <= dealer limit
"""
    )

    print("\n" + "=" * 110)
    print("INPUT REVIEW COMPLETE -- RUNNING OPTIMIZER")
    print("=" * 110)


# ============================================================================
# BUILD DECISION VARIABLES
# ============================================================================


def build_decision_variables(
    assets: pd.DataFrame,
    margin_terms: pd.DataFrame,
    repo_terms: pd.DataFrame,
    cash_funding_sources: pd.DataFrame,
    *,
    allow_repo: bool = True,
    allow_fx_swap: bool = True,
) -> pd.DataFrame:
    """Create one row per LP decision variable."""

    asset_costs = assets[
        [
            "asset",
            "opportunity_cost_bps",
        ]
    ]

    # ------------------------------------------------------------------------
    # Margin variables
    # ------------------------------------------------------------------------

    margin_variables = margin_terms.merge(
        asset_costs,
        on="asset",
        how="left",
        validate="many_to_one",
    )

    margin_variables["variable"] = (
        "MARGIN::"
        + margin_variables["counterparty"]
        + "::"
        + margin_variables["asset"]
    )

    margin_variables["kind"] = "margin_collateral"

    margin_variables["destination"] = (
        margin_variables["counterparty"]
    )

    margin_variables["cash_use_per_unit"] = np.where(
        margin_variables["asset"].eq("USD_CASH"),
        1.0,
        0.0,
    )

    margin_variables["cash_proceeds_per_unit"] = 0.0

    margin_variables["effective_margin_per_unit"] = (
        1.0 - margin_variables["haircut"]
    )

    margin_variables["stable_cash_per_unit"] = 0.0

    margin_variables["upper_bound"] = np.inf

    margin_variables[
        "opportunity_cost_bps_per_decision_unit"
    ] = margin_variables[
        "opportunity_cost_bps"
    ]

    margin_variables[
        "transfer_cost_bps_per_decision_unit"
    ] = margin_variables[
        "transfer_cost_bps"
    ]

    margin_variables[
        "financing_cost_bps_per_decision_unit"
    ] = 0.0

    margin_variables[
        "cost_bps_per_decision_unit"
    ] = (
        margin_variables[
            "opportunity_cost_bps_per_decision_unit"
        ]
        + margin_variables[
            "transfer_cost_bps_per_decision_unit"
        ]
        + margin_variables[
            "financing_cost_bps_per_decision_unit"
        ]
    )

    margin_variables["dealer"] = None

    margin_variables["tenor_days"] = 0

    # ------------------------------------------------------------------------
    # Repo variables
    # ------------------------------------------------------------------------

    repo_variables = repo_terms.merge(
        asset_costs,
        on="asset",
        how="left",
        validate="many_to_one",
    )

    repo_variables["variable"] = (
        "REPO::"
        + repo_variables["route"]
    )

    repo_variables["kind"] = "repo_funding"

    repo_variables["destination"] = (
        repo_variables["route"]
    )

    repo_variables["cash_use_per_unit"] = 0.0

    repo_variables["cash_proceeds_per_unit"] = (
        1.0 - repo_variables["haircut"]
    )

    repo_variables["effective_margin_per_unit"] = 0.0

    repo_variables["stable_cash_per_unit"] = np.where(
        repo_variables["tenor_days"] >= 30,
        repo_variables["cash_proceeds_per_unit"],
        0.0,
    )

    repo_variables["upper_bound"] = np.where(
        allow_repo,
        repo_variables["maximum_collateral_usd_m"],
        0.0,
    )

    repo_variables[
        "opportunity_cost_bps_per_decision_unit"
    ] = repo_variables[
        "opportunity_cost_bps"
    ]

    repo_variables[
        "transfer_cost_bps_per_decision_unit"
    ] = 0.0

    # Repo rates apply to cash proceeds.  The LP decision is collateral
    # market value, so convert the financing component to the same unit.
    repo_variables[
        "financing_cost_bps_per_decision_unit"
    ] = (
        repo_variables["repo_cash_cost_bps"]
        * repo_variables["cash_proceeds_per_unit"]
    )

    repo_variables[
        "cost_bps_per_decision_unit"
    ] = (
        repo_variables[
            "opportunity_cost_bps_per_decision_unit"
        ]
        + repo_variables[
            "transfer_cost_bps_per_decision_unit"
        ]
        + repo_variables[
            "financing_cost_bps_per_decision_unit"
        ]
    )

    repo_variables["counterparty"] = None

    # ------------------------------------------------------------------------
    # Other cash funding variables
    # ------------------------------------------------------------------------

    cash_variables = cash_funding_sources.copy()

    cash_variables["variable"] = (
        "CASH_FUNDING::"
        + cash_variables["source"]
    )

    cash_variables["kind"] = "cash_funding"

    cash_variables["asset"] = None

    cash_variables["destination"] = (
        cash_variables["source"]
    )

    cash_variables["dealer"] = None

    cash_variables["haircut"] = 0.0

    cash_variables["cash_use_per_unit"] = 0.0

    cash_variables["cash_proceeds_per_unit"] = 1.0

    cash_variables["effective_margin_per_unit"] = 0.0

    cash_variables["stable_cash_per_unit"] = np.where(
        cash_variables["tenor_days"] >= 30,
        1.0,
        0.0,
    )

    cash_variables["upper_bound"] = (
        cash_variables["maximum_cash_usd_m"]
    )

    if not allow_fx_swap:
        cash_variables.loc[
            cash_variables["funding_type"].eq("fx_swap"),
            "upper_bound",
        ] = 0.0

    cash_variables[
        "opportunity_cost_bps_per_decision_unit"
    ] = 0.0

    cash_variables[
        "transfer_cost_bps_per_decision_unit"
    ] = 0.0

    cash_variables[
        "financing_cost_bps_per_decision_unit"
    ] = cash_variables[
        "all_in_cost_bps"
    ]

    cash_variables[
        "cost_bps_per_decision_unit"
    ] = (
        cash_variables[
            "opportunity_cost_bps_per_decision_unit"
        ]
        + cash_variables[
            "transfer_cost_bps_per_decision_unit"
        ]
        + cash_variables[
            "financing_cost_bps_per_decision_unit"
        ]
    )

    cash_variables["counterparty"] = None

    # ------------------------------------------------------------------------
    # Combine
    # ------------------------------------------------------------------------

    common_columns = [
        "variable",
        "kind",
        "asset",
        "destination",
        "counterparty",
        "dealer",
        "haircut",
        "tenor_days",
        "cash_use_per_unit",
        "cash_proceeds_per_unit",
        "effective_margin_per_unit",
        "stable_cash_per_unit",
        "opportunity_cost_bps_per_decision_unit",
        "transfer_cost_bps_per_decision_unit",
        "financing_cost_bps_per_decision_unit",
        "cost_bps_per_decision_unit",
        "upper_bound",
    ]

    variables = pd.concat(
        [
            margin_variables[common_columns],
            repo_variables[common_columns],
            cash_variables[common_columns],
        ],
        ignore_index=True,
    )

    variables["decision_index"] = np.arange(
        len(variables)
    )

    return variables


# ============================================================================
# PRINT OPTIMIZATION-ENGINE INPUTS
# ============================================================================


def print_optimization_engine_inputs(
    variables: pd.DataFrame,
    rows: list[np.ndarray],
    limits: list[float],
    names: list[str],
    objective: np.ndarray,
    bounds: list[tuple[float, float | None]],
) -> None:
    """Print the exact decision vector and inequality rows sent to linprog."""

    decision_units = {
        "margin_collateral":
            "collateral market value",
        "repo_funding":
            "collateral market value",
        "cash_funding":
            "cash raised",
    }

    engine_variables = variables.loc[
        :,
        [
            "decision_index",
            "variable",
            "kind",
            "opportunity_cost_bps_per_decision_unit",
            "transfer_cost_bps_per_decision_unit",
            "financing_cost_bps_per_decision_unit",
            "cost_bps_per_decision_unit",
        ],
    ].copy()

    engine_variables["decision_unit"] = (
        engine_variables["kind"].map(
            decision_units
        )
    )

    engine_variables["objective_c"] = objective

    engine_variables["lower_bound_usd_m"] = [
        lower
        for lower, _ in bounds
    ]

    engine_variables["upper_bound_usd_m"] = [
        "unbounded"
        if upper is None
        else f"{upper:,.4f}"
        for _, upper in bounds
    ]

    engine_variables = engine_variables.rename(
        columns={
            "decision_index": "index",
            "variable": "decision_variable",
            "kind": "decision_type",
            "opportunity_cost_bps_per_decision_unit":
                "opportunity_bps",
            "transfer_cost_bps_per_decision_unit":
                "transfer_bps",
            "financing_cost_bps_per_decision_unit":
                "financing_bps",
            "cost_bps_per_decision_unit":
                "total_objective_bps",
        }
    )[
        [
            "index",
            "decision_variable",
            "decision_type",
            "decision_unit",
            "opportunity_bps",
            "transfer_bps",
            "financing_bps",
            "total_objective_bps",
            "objective_c",
            "lower_bound_usd_m",
            "upper_bound_usd_m",
        ]
    ]

    clean_rows = [
        np.where(
            np.abs(row) <= 1e-12,
            0.0,
            row,
        )
        for row in rows
    ]

    coefficient_vectors = [
        np.array2string(
            row,
            separator=", ",
            formatter={
                "float_kind":
                    lambda value: f"{value:.4f}"
            },
            max_line_width=1_000,
        )
        for row in clean_rows
    ]

    engine_constraints = pd.DataFrame(
        {
            "row": np.arange(len(clean_rows)),
            "constraint": names,
            "coefficients_for_x0_to_x16":
                coefficient_vectors,
            "relation": "<=",
            "rhs_limit": limits,
        }
    )

    pd.set_option("display.width", 260)
    pd.set_option("display.max_columns", 40)
    pd.set_option(
        "display.float_format",
        lambda value: f"{value:,.6f}",
    )

    print("\n" + "#" * 110)
    print("OPTIMIZATION ENGINE INPUTS")
    print("#" * 110)

    print("\n" + "=" * 110)
    print("A. DECISION VECTOR x -- ALL 17 VARIABLES")
    print("=" * 110)
    print(
        "linprog minimizes c @ x.  "
        "objective_c = objective cost in bps / 10,000."
    )
    print(
        "All cost columns are annualized bps per decision unit; "
        "total_objective_bps = opportunity + transfer + financing."
    )
    print(
        "For repo, financing_bps equals the quoted repo rate multiplied "
        "by cash proceeds per collateral unit."
    )
    print(
        engine_variables.to_string(index=False)
    )

    print("\n" + "=" * 110)
    print("B. INEQUALITY ROWS -- ALL 11 CONSTRAINTS")
    print("=" * 110)
    print(
        f"A_ub shape: ({len(clean_rows)}, "
        f"{len(variables)})"
    )
    print(
        "Each row means: coefficient vector @ x <= rhs_limit.  "
        "Coefficient order is x[0] through x[16] from the table above."
    )
    print(
        engine_constraints.to_string(index=False)
    )

    print("\nDecoded nonzero terms:")

    for row_index, (
        constraint_name,
        row,
        limit,
    ) in enumerate(
        zip(
            names,
            clean_rows,
            limits,
        )
    ):
        terms = [
            f"{coefficient:+.4f}*x[{variable_index}]"
            for variable_index, coefficient in enumerate(row)
            if abs(coefficient) > 1e-12
        ]

        left_hand_side = " ".join(terms)

        if left_hand_side.startswith("+"):
            left_hand_side = left_hand_side[1:]

        print(
            f"row[{row_index:02d}] "
            f"{constraint_name}: "
            f"{left_hand_side} <= {limit:,.4f}"
        )

    print("\n" + "=" * 110)
    print("INPUT ASSEMBLY COMPLETE -- CALLING scipy.optimize.linprog")
    print("=" * 110)


# ============================================================================
# SOLVE LP
# ============================================================================


def solve_joint_problem(
    assets: pd.DataFrame,
    margin_requirements: pd.DataFrame,
    margin_terms: pd.DataFrame,
    repo_terms: pd.DataFrame,
    repo_dealer_limits: pd.DataFrame,
    cash_funding_sources: pd.DataFrame,
    cash_policy: CashPolicy,
    *,
    allow_repo: bool = True,
    allow_fx_swap: bool = True,
    print_engine_inputs: bool = False,
) -> JointSolution:
    """Solve the joint funding and collateral LP."""

    variables = build_decision_variables(
        assets,
        margin_terms,
        repo_terms,
        cash_funding_sources,
        allow_repo=allow_repo,
        allow_fx_swap=allow_fx_swap,
    )

    rows: list[np.ndarray] = []
    limits: list[float] = []
    names: list[str] = []
    categories: list[str] = []

    # ------------------------------------------------------------------------
    # 1. Security inventory
    # ------------------------------------------------------------------------

    security_assets = assets.loc[
        assets["is_security"]
    ]

    for asset_row in security_assets.itertuples(
        index=False
    ):
        uses_asset = (
            variables["asset"]
            .eq(asset_row.asset)
            .to_numpy(float)
        )

        rows.append(uses_asset)

        limits.append(
            float(asset_row.usable_inventory_usd_m)
        )

        names.append(
            f"inventory::{asset_row.asset}"
        )

        categories.append("inventory")

    # ------------------------------------------------------------------------
    # 2. Margin coverage
    # ------------------------------------------------------------------------

    for margin_row in margin_requirements.itertuples(
        index=False
    ):
        is_margin_for_counterparty = (
            variables["kind"].eq(
                "margin_collateral"
            )
            & variables["counterparty"].eq(
                margin_row.counterparty
            )
        ).to_numpy(float)

        coverage = (
            is_margin_for_counterparty
            * variables[
                "effective_margin_per_unit"
            ].to_numpy()
        )

        rows.append(-coverage)

        limits.append(
            -float(
                margin_row.required_margin_usd_m
            )
        )

        names.append(
            f"margin::{margin_row.counterparty}"
        )

        categories.append("margin")

    # ------------------------------------------------------------------------
    # 3. Cash balance
    # ------------------------------------------------------------------------

    cash_balance_row = (
        variables[
            "cash_use_per_unit"
        ].to_numpy()
        -
        variables[
            "cash_proceeds_per_unit"
        ].to_numpy()
    )

    rows.append(cash_balance_row)

    limits.append(
        cash_policy.opening_cash_usd_m
        - cash_policy.required_non_margin_cash_usd_m
    )

    names.append(
        "cash_balance::USD"
    )

    categories.append(
        "cash_balance"
    )

    # ------------------------------------------------------------------------
    # 4. Stable funding
    # ------------------------------------------------------------------------

    rows.append(
        -variables[
            "stable_cash_per_unit"
        ].to_numpy()
    )

    limits.append(
        -cash_policy.minimum_stable_funding_usd_m
    )

    names.append(
        "stable_funding::30_plus_days"
    )

    categories.append(
        "stable_funding"
    )

    # ------------------------------------------------------------------------
    # 5. Repo dealer capacity
    # ------------------------------------------------------------------------

    for dealer_row in repo_dealer_limits.itertuples(
        index=False
    ):
        is_dealer = (
            variables["dealer"]
            .eq(dealer_row.dealer)
            .to_numpy(float)
        )

        dealer_cash_proceeds = (
            is_dealer
            * variables[
                "cash_proceeds_per_unit"
            ].to_numpy()
        )

        rows.append(
            dealer_cash_proceeds
        )

        limits.append(
            float(
                dealer_row.maximum_cash_proceeds_usd_m
            )
        )

        names.append(
            f"repo_dealer_capacity::{dealer_row.dealer}"
        )

        categories.append(
            "repo_dealer_capacity"
        )

    # ------------------------------------------------------------------------
    # Objective
    # ------------------------------------------------------------------------

    objective = (
        variables[
            "cost_bps_per_decision_unit"
        ].to_numpy()
        / 10_000.0
    )

    bounds = [
        (
            0.0,
            None
            if np.isinf(upper)
            else float(upper),
        )
        for upper in variables["upper_bound"]
    ]

    if print_engine_inputs:
        print_optimization_engine_inputs(
            variables,
            rows,
            limits,
            names,
            objective,
            bounds,
        )

    # ------------------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------------------

    result = linprog(
        c=objective,
        A_ub=np.vstack(rows),
        b_ub=np.asarray(limits),
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            "Joint Treasury optimization failed: "
            f"{result.message}"
        )

    # ------------------------------------------------------------------------
    # Store solution
    # ------------------------------------------------------------------------

    variables = variables.copy()

    variables["decision_usd_m"] = result.x

    variables[
        "effective_margin_usd_m"
    ] = (
        variables["decision_usd_m"]
        * variables[
            "effective_margin_per_unit"
        ]
    )

    variables[
        "cash_raised_usd_m"
    ] = (
        variables["decision_usd_m"]
        * variables[
            "cash_proceeds_per_unit"
        ]
    )

    variables[
        "annual_cost_usd_m"
    ] = (
        result.x
        * objective
    )

    variables["effective_cost_bps"] = np.where(
        variables["kind"].eq(
            "margin_collateral"
        ),
        (
            variables[
                "cost_bps_per_decision_unit"
            ]
            /
            variables[
                "effective_margin_per_unit"
            ]
        ),
        (
            variables[
                "cost_bps_per_decision_unit"
            ]
            /
            variables[
                "cash_proceeds_per_unit"
            ].replace(0.0, np.nan)
        ),
    )

    constraints = pd.DataFrame(
        {
            "constraint": names,
            "category": categories,
            "limit": limits,
            "slack": result.ineqlin.residual,
            "solver_marginal": result.ineqlin.marginals,
        }
    )

    return JointSolution(
        result=result,
        variables=variables,
        constraints=constraints,
    )


# ============================================================================
# SUMMARIZE RESULTS
# ============================================================================


def summarize_solution(
    solution: JointSolution,
    assets: pd.DataFrame,
    margin_requirements: pd.DataFrame,
    cash_policy: CashPolicy,
) -> dict[str, float]:
    """Print economically meaningful outputs from the LP."""

    variables = solution.variables

    active = variables.loc[
        variables["decision_usd_m"] > 1e-8
    ].copy()

    # ------------------------------------------------------------------------
    # Complete decision vector, including zero-valued decisions
    # ------------------------------------------------------------------------

    all_decisions = variables.loc[
        :,
        [
            "decision_index",
            "variable",
            "kind",
            "decision_usd_m",
            "opportunity_cost_bps_per_decision_unit",
            "transfer_cost_bps_per_decision_unit",
            "financing_cost_bps_per_decision_unit",
            "cost_bps_per_decision_unit",
        ],
    ].copy()

    # Replace tiny numerical solver values with an exact zero for display.
    zero_decision_mask = (
        all_decisions["decision_usd_m"].abs() <= 1e-8
    )

    all_decisions.loc[
        zero_decision_mask,
        "decision_usd_m",
    ] = 0.0

    # With decision values in USD millions and costs in annualized bps:
    # annual cost in USD = decision_usd_m * component_bps * 100.
    cost_components = {
        "opportunity_cost_bps_per_decision_unit":
            "opportunity_cost_usd",
        "transfer_cost_bps_per_decision_unit":
            "transfer_cost_usd",
        "financing_cost_bps_per_decision_unit":
            "financing_cost_usd",
    }

    for bps_column, usd_column in cost_components.items():
        all_decisions[usd_column] = (
            all_decisions["decision_usd_m"]
            * all_decisions[bps_column]
            * 100.0
        )

    all_decisions["total_annual_cost_usd"] = (
        all_decisions[
            list(cost_components.values())
        ].sum(axis=1)
    )

    all_decisions.loc[
        zero_decision_mask,
        [
            *cost_components.values(),
            "total_annual_cost_usd",
        ],
    ] = 0.0

    all_decisions["decision_unit"] = (
        all_decisions["kind"].map(
            {
                "margin_collateral":
                    "collateral market value",
                "repo_funding":
                    "collateral market value",
                "cash_funding":
                    "cash raised",
            }
        )
    )

    all_decisions["status"] = np.where(
        all_decisions["decision_usd_m"] > 1e-8,
        "ACTIVE",
        "ZERO",
    )

    all_decisions = all_decisions.rename(
        columns={
            "decision_index": "index",
            "variable": "decision_variable",
            "kind": "decision_type",
            "decision_usd_m": "optimal_value_usd_m",
            "cost_bps_per_decision_unit":
                "total_objective_bps",
        }
    )[
        [
            "index",
            "decision_variable",
            "decision_type",
            "decision_unit",
            "optimal_value_usd_m",
            "total_objective_bps",
            "opportunity_cost_usd",
            "transfer_cost_usd",
            "financing_cost_usd",
            "total_annual_cost_usd",
            "status",
        ]
    ]

    total_opportunity_cost_usd = float(
        all_decisions[
            "opportunity_cost_usd"
        ].sum()
    )

    total_transfer_cost_usd = float(
        all_decisions[
            "transfer_cost_usd"
        ].sum()
    )

    total_financing_cost_usd = float(
        all_decisions[
            "financing_cost_usd"
        ].sum()
    )

    total_all_decision_cost_usd = float(
        all_decisions[
            "total_annual_cost_usd"
        ].sum()
    )

    solver_objective_usd = float(
        solution.result.fun
        * 1_000_000.0
    )

    # ------------------------------------------------------------------------
    # Margin plan
    # ------------------------------------------------------------------------

    margin_plan = active.loc[
        active["kind"].eq(
            "margin_collateral"
        ),
        [
            "counterparty",
            "asset",
            "haircut",
            "decision_usd_m",
            "effective_margin_usd_m",
            "effective_cost_bps",
            "annual_cost_usd_m",
        ],
    ].rename(
        columns={
            "decision_usd_m":
                "posted_market_value_usd_m"
        }
    )

    # ------------------------------------------------------------------------
    # Funding plan
    # ------------------------------------------------------------------------

    funding_plan = active.loc[
        active["kind"].isin(
            [
                "repo_funding",
                "cash_funding",
            ]
        ),
        [
            "kind",
            "destination",
            "asset",
            "tenor_days",
            "decision_usd_m",
            "cash_raised_usd_m",
            "effective_cost_bps",
            "annual_cost_usd_m",
        ],
    ].rename(
        columns={
            "decision_usd_m":
                "decision_or_collateral_usd_m"
        }
    )

    # ------------------------------------------------------------------------
    # Margin coverage
    # ------------------------------------------------------------------------

    margin_coverage = (
        margin_plan.groupby(
            "counterparty",
            as_index=False,
        )[
            "effective_margin_usd_m"
        ]
        .sum()
        .merge(
            margin_requirements,
            on="counterparty",
            how="right",
            validate="one_to_one",
        )
    )

    margin_coverage[
        "effective_margin_usd_m"
    ] = margin_coverage[
        "effective_margin_usd_m"
    ].fillna(0.0)

    margin_coverage[
        "excess_margin_usd_m"
    ] = (
        margin_coverage[
            "effective_margin_usd_m"
        ]
        -
        margin_coverage[
            "required_margin_usd_m"
        ]
    )

    # ------------------------------------------------------------------------
    # Security usage
    # ------------------------------------------------------------------------

    security_usage = (
        active.loc[
            active["asset"].notna()
            & active["asset"].ne("USD_CASH")
        ]
        .groupby(
            [
                "asset",
                "kind",
            ],
            as_index=False,
        )["decision_usd_m"]
        .sum()
        .pivot(
            index="asset",
            columns="kind",
            values="decision_usd_m",
        )
        .fillna(0.0)
        .reset_index()
        .merge(
            assets.loc[
                assets["is_security"],
                [
                    "asset",
                    "usable_inventory_usd_m",
                ],
            ],
            on="asset",
            how="right",
            validate="one_to_one",
        )
    )

    for column in [
        "margin_collateral",
        "repo_funding",
    ]:
        if column not in security_usage:
            security_usage[column] = 0.0

    security_usage[
        [
            "margin_collateral",
            "repo_funding",
        ]
    ] = security_usage[
        [
            "margin_collateral",
            "repo_funding",
        ]
    ].fillna(0.0)

    security_usage[
        "total_used_usd_m"
    ] = (
        security_usage[
            "margin_collateral"
        ]
        +
        security_usage[
            "repo_funding"
        ]
    )

    security_usage[
        "unused_usd_m"
    ] = (
        security_usage[
            "usable_inventory_usd_m"
        ]
        -
        security_usage[
            "total_used_usd_m"
        ]
    )

    # ------------------------------------------------------------------------
    # Cash
    # ------------------------------------------------------------------------

    cash_margin_posted = float(
        active.loc[
            active["kind"].eq(
                "margin_collateral"
            )
            & active["asset"].eq(
                "USD_CASH"
            ),
            "decision_usd_m",
        ].sum()
    )

    repo_cash = float(
        active.loc[
            active["kind"].eq(
                "repo_funding"
            ),
            "cash_raised_usd_m",
        ].sum()
    )

    other_funding_cash = float(
        active.loc[
            active["kind"].eq(
                "cash_funding"
            ),
            "cash_raised_usd_m",
        ].sum()
    )

    ending_cash = (
        cash_policy.opening_cash_usd_m
        + repo_cash
        + other_funding_cash
        - cash_margin_posted
        - cash_policy.settlement_cash_need_usd_m
    )

    # ------------------------------------------------------------------------
    # Costs
    # ------------------------------------------------------------------------

    collateral_cost = float(
        active.loc[
            active["kind"].eq(
                "margin_collateral"
            ),
            "annual_cost_usd_m",
        ].sum()
    )

    funding_cost = float(
        active.loc[
            active["kind"].isin(
                [
                    "repo_funding",
                    "cash_funding",
                ]
            ),
            "annual_cost_usd_m",
        ].sum()
    )

    total_cash_raised = (
        repo_cash
        + other_funding_cash
    )

    if total_cash_raised > 0:
        weighted_funding_cost_bps = (
            funding_cost
            / total_cash_raised
            * 10_000.0
        )
    else:
        weighted_funding_cost_bps = np.nan

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 30)
    pd.set_option(
        "display.float_format",
        lambda value: f"{value:,.4f}",
    )

    # ------------------------------------------------------------------------
    # Complete optimal decision vector
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("OPTIMAL SOLUTION -- ALL 17 DECISION VARIABLES")
    print("USD millions; zero-valued decisions are intentionally included")
    print(
        "Each annual USD cost component = optimal value in USD millions "
        "x component cost in bps x 100"
    )
    print(
        "A zero component means that cost type is not separately modeled "
        "for that decision."
    )
    print("=" * 110)

    print(
        all_decisions.to_string(index=False)
    )

    print("-" * 110)
    print(
        "TOTAL OPPORTUNITY COST:                         "
        f"${total_opportunity_cost_usd:,.2f}"
    )
    print(
        "TOTAL TRANSFER COST:                            "
        f"${total_transfer_cost_usd:,.2f}"
    )
    print(
        "TOTAL FINANCING COST:                           "
        f"${total_financing_cost_usd:,.2f}"
    )
    print(
        "SUM OF ANNUAL COSTS ACROSS ALL 17 DECISIONS: "
        f"${total_all_decision_cost_usd:,.2f}"
    )
    print(
        "SOLVER OBJECTIVE VALUE:                         "
        f"${solver_objective_usd:,.2f}"
    )
    print(
        "RECONCILIATION DIFFERENCE:                      "
        f"${total_all_decision_cost_usd - solver_objective_usd:,.8f}"
    )

    # ------------------------------------------------------------------------
    # Optimal margin collateral
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("OPTIMAL MARGIN COLLATERAL")
    print("USD millions")
    print("=" * 110)

    print(
        margin_plan.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Margin coverage
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("MARGIN COVERAGE CHECK")
    print("=" * 110)

    print(
        margin_coverage.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Funding plan
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("OPTIMAL FUNDING PLAN")
    print("USD millions")
    print("=" * 110)

    print(
        funding_plan.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Inventory
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("SHARED SECURITY INVENTORY CHECK")
    print("=" * 110)

    print(
        security_usage[
            [
                "asset",
                "margin_collateral",
                "repo_funding",
                "total_used_usd_m",
                "usable_inventory_usd_m",
                "unused_usd_m",
            ]
        ].to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Cash reconciliation
    # ------------------------------------------------------------------------

    cash_reconciliation = pd.DataFrame(
        {
            "cash_item": [
                "Opening USD cash",
                "Repo cash raised",
                "Other / FX cash raised",
                "Cash posted for margin",
                "Settlement cash need",
                "Ending USD cash",
                "Minimum liquidity buffer",
            ],
            "usd_m": [
                cash_policy.opening_cash_usd_m,
                repo_cash,
                other_funding_cash,
                -cash_margin_posted,
                -cash_policy.settlement_cash_need_usd_m,
                ending_cash,
                cash_policy.minimum_liquidity_buffer_usd_m,
            ],
        }
    )

    print("\n" + "=" * 110)
    print("USD CASH RECONCILIATION")
    print("=" * 110)

    print(
        cash_reconciliation.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Cost summary
    # ------------------------------------------------------------------------

    cost_summary = pd.DataFrame(
        {
            "cost_component": [
                "Margin-collateral economic cost",
                "Funding economic cost",
                "Total annual economic cost",
            ],
            "annual_cost_usd_m": [
                collateral_cost,
                funding_cost,
                solution.result.fun,
            ],
            "annual_cost_usd": [
                collateral_cost * 1_000_000,
                funding_cost * 1_000_000,
                solution.result.fun * 1_000_000,
            ],
        }
    )

    print("\n" + "=" * 110)
    print("COST SUMMARY")
    print("=" * 110)

    print(
        cost_summary.to_string(index=False)
    )

    print(
        f"\nWeighted effective cost of cash raised: "
        f"{weighted_funding_cost_bps:,.2f} bps"
    )

    # ------------------------------------------------------------------------
    # Marginal costs
    # ------------------------------------------------------------------------

    useful_duals = (
        solution.constraints.loc[
            solution.constraints[
                "category"
            ].isin(
                [
                    "margin",
                    "cash_balance",
                    "stable_funding",
                ]
            )
        ]
        .copy()
    )

    useful_duals[
        "marginal_cost_bps"
    ] = (
        -useful_duals[
            "solver_marginal"
        ]
        * 10_000.0
    )

    print("\n" + "=" * 110)
    print("MARGINAL COST OF USD 1mm MORE REQUIREMENT")
    print("=" * 110)

    print(
        useful_duals[
            [
                "constraint",
                "slack",
                "marginal_cost_bps",
            ]
        ].to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Scarce capacity
    # ------------------------------------------------------------------------

    scarce_capacity_duals = (
        solution.constraints.loc[
            solution.constraints[
                "category"
            ].isin(
                [
                    "inventory",
                    "repo_dealer_capacity",
                ]
            )
        ]
        .copy()
    )

    scarce_capacity_duals[
        "value_of_one_more_usd_1mm_bps"
    ] = (
        -scarce_capacity_duals[
            "solver_marginal"
        ]
        * 10_000.0
    )

    print("\n" + "=" * 110)
    print("VALUE OF USD 1mm MORE INVENTORY OR DEALER CAPACITY")
    print("=" * 110)

    print(
        scarce_capacity_duals[
            [
                "constraint",
                "slack",
                "value_of_one_more_usd_1mm_bps",
            ]
        ].to_string(index=False)
    )

    return {
        "total_annual_cost_usd_m":
            float(solution.result.fun),

        "collateral_cost_usd_m":
            collateral_cost,

        "funding_cost_usd_m":
            funding_cost,

        "total_cash_raised_usd_m":
            total_cash_raised,

        "weighted_funding_cost_bps":
            weighted_funding_cost_bps,

        "ending_cash_usd_m":
            ending_cash,
    }


# ============================================================================
# MAIN
# ============================================================================


def main() -> None:

    (
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
    ) = make_synthetic_inputs()

    print("\n" + "#" * 110)
    print("JOINT FUNDING AND COLLATERAL OPTIMIZATION")
    print("#" * 110)

    print(
        """
All data are synthetic.
All monetary amounts are USD millions.
All rates and costs are annualized basis points.

Purpose:

    Illustrate how collateral allocation and funding efficiency
    can be optimized jointly.
"""
    )

    # ------------------------------------------------------------------------
    # Print inputs
    # ------------------------------------------------------------------------

    print_model_inputs(
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
    )

    # ------------------------------------------------------------------------
    # Solve joint optimum
    # ------------------------------------------------------------------------

    joint_solution = solve_joint_problem(
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
        print_engine_inputs=True,
    )

    print("\n" + "#" * 110)
    print("OPTIMIZATION SUCCESSFUL")
    print("#" * 110)

    print(
        f"Solver status:      "
        f"{joint_solution.result.message}"
    )

    print(
        f"Solver iterations:  "
        f"{joint_solution.result.nit}"
    )

    print(
        f"Decision variables: "
        f"{len(joint_solution.variables)}"
    )

    print(
        f"LP constraints:     "
        f"{len(joint_solution.constraints)}"
    )

    joint_summary = summarize_solution(
        joint_solution,
        assets,
        margin_requirements,
        cash_policy,
    )

    # ------------------------------------------------------------------------
    # Counterfactual: no repo
    # ------------------------------------------------------------------------

    print("\n" + "#" * 110)
    print("RESEARCH COUNTERFACTUAL: WHAT IF REPO WERE UNAVAILABLE?")
    print("#" * 110)

    print(
        """
We re-optimize the entire Treasury problem with repo disabled.

This lets collateral allocation and other funding sources adjust
optimally.

The increase in minimum economic cost represents the modeled value
of having repo available.
"""
    )

    no_repo_solution = solve_joint_problem(
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
        allow_repo=False,
    )

    no_repo_total_cost = float(
        no_repo_solution.result.fun
    )

    annual_savings = (
        no_repo_total_cost
        - joint_summary[
            "total_annual_cost_usd_m"
        ]
    )

    comparison = pd.DataFrame(
        {
            "scenario": [
                "Joint optimum",
                "Counterfactual: repo unavailable",
            ],
            "annual_economic_cost_usd_m": [
                joint_summary[
                    "total_annual_cost_usd_m"
                ],
                no_repo_total_cost,
            ],
            "annual_economic_cost_usd": [
                joint_summary[
                    "total_annual_cost_usd_m"
                ]
                * 1_000_000,

                no_repo_total_cost
                * 1_000_000,
            ],
        }
    )

    print("\nRESEARCH COUNTERFACTUAL")

    print(
        comparison.to_string(index=False)
    )

    print(
        f"\nEstimated annual value of repo availability: "
        f"${annual_savings * 1_000_000:,.0f}"
    )

    # ------------------------------------------------------------------------
    # Final interpretation
    # ------------------------------------------------------------------------

    print("\n" + "#" * 110)
    print("FINAL ECONOMIC INTERPRETATION")
    print("#" * 110)

    print(
        f"""
Optimal annual economic cost:
    ${joint_summary["total_annual_cost_usd_m"] * 1_000_000:,.0f}

Collateral allocation cost:
    ${joint_summary["collateral_cost_usd_m"] * 1_000_000:,.0f}

Funding cost:
    ${joint_summary["funding_cost_usd_m"] * 1_000_000:,.0f}

Total external cash raised:
    ${joint_summary["total_cash_raised_usd_m"]:,.2f}m

Weighted effective funding cost:
    {joint_summary["weighted_funding_cost_bps"]:,.2f} bps

Ending USD liquidity:
    ${joint_summary["ending_cash_usd_m"]:,.2f}m


CORE TREASURY QUESTION
----------------------

For every scarce security, the optimizer effectively asks:

    Where does this asset create the most economic value?

Possible uses:

    1. Post it as margin collateral
    2. Pledge it into repo and raise USD cash
    3. Leave it unencumbered


The key linking constraint is:

    MARGIN USE
    +
    REPO USE

        <= AVAILABLE SECURITY INVENTORY


That is what makes this a joint funding-efficiency and
collateral-allocation optimization problem.
"""
    )


if __name__ == "__main__":
    main()



##############################################################################################################
JOINT FUNDING AND COLLATERAL OPTIMIZATION
##############################################################################################################

All data are synthetic.
All monetary amounts are USD millions.
All rates and costs are annualized basis points.

Purpose:

    Illustrate how collateral allocation and funding efficiency
    can be optimized jointly.


1. PROBLEM TO SOLVE

We need to satisfy BOTH:

    A. Margin requirements at CCPs / prime brokers
    B. USD liquidity requirements

while minimizing annual economic cost.

The main trade-off is:

                         SECURITY INVENTORY
                                |
                --------------------------------
                |                              |
                v                              v
        POST FOR MARGIN                 PLEDGE INTO REPO
                |                   

In [ ]:
"""Joint funding-efficiency and collateral-allocation optimization example.

This synthetic interview-preparation example illustrates a central Treasury
optimization problem:

    How should a global trading firm meet margin and USD liquidity needs at
    minimum risk-adjusted annual cost, when the same security can either be
    posted for margin or pledged to raise cash through repo, but not both?

The model jointly chooses:

1. Margin collateral posted to each counterparty.
2. Securities pledged into repo to raise USD cash.
3. Other cash-funding routes, including an EUR-to-USD FX swap.

It enforces:

* counterparty eligibility and haircuts;
* security-inventory conservation (no double use of collateral);
* a USD cash balance for settlements, margin, and liquidity buffer;
* funding-route and repo-dealer capacities; and
* a minimum amount of stable (30+ day) external funding.

All amounts are USD millions and all costs are annualized basis points.

The data are deliberately synthetic and do not describe HRT's actual
platform, positions, counterparties, or financing arrangements.

Install:
    python -m pip install numpy pandas scipy

Run:
    python joint_treasury_funding_collateral_optimization.py
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import OptimizeResult, linprog


# ============================================================================
# DATA CLASSES
# ============================================================================


@dataclass(frozen=True)
class CashPolicy:
    """USD cash requirements that the joint optimizer must satisfy."""

    opening_cash_usd_m: float
    settlement_cash_need_usd_m: float
    minimum_liquidity_buffer_usd_m: float
    minimum_stable_funding_usd_m: float
    stable_tenor_days: int = 30

    @property
    def required_non_margin_cash_usd_m(self) -> float:
        """Cash retained for settlements and the end-of-day buffer."""
        return (
            self.settlement_cash_need_usd_m
            + self.minimum_liquidity_buffer_usd_m
        )


@dataclass
class JointSolution:
    """Solver output plus tables needed to explain the recommendation."""

    result: OptimizeResult
    variables: pd.DataFrame
    constraints: pd.DataFrame


# ============================================================================
# SYNTHETIC INPUT DATA
# ============================================================================


def make_synthetic_inputs() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    CashPolicy,
]:
    """Create a small synthetic global funding and collateral data set."""

    # ------------------------------------------------------------------------
    # Available assets
    # ------------------------------------------------------------------------

    assets = pd.DataFrame(
        {
            "asset": [
                "USD_CASH",
                "T_BILL",
                "UST_2Y",
                "UST_10Y",
                "CORP_BOND",
            ],
            "is_security": [
                False,
                True,
                True,
                True,
                True,
            ],
            "usable_inventory_usd_m": [
                np.nan,
                90.0,
                80.0,
                70.0,
                50.0,
            ],
            "opportunity_cost_bps": [
                70.0,
                8.0,
                14.0,
                25.0,
                40.0,
            ],
        }
    )

    # ------------------------------------------------------------------------
    # Margin requirements
    # ------------------------------------------------------------------------

    margin_requirements = pd.DataFrame(
        {
            "counterparty": [
                "CCP_A",
                "PB_B_CASH_ONLY",
                "CCP_C",
            ],
            "required_margin_usd_m": [
                132.6428,
                64.8476,
                97.2714,
            ],
        }
    )

    # ------------------------------------------------------------------------
    # Margin collateral terms
    # ------------------------------------------------------------------------

    margin_terms = pd.DataFrame(
        [
            ("CCP_A", "USD_CASH", 0.000, 0.2),
            ("CCP_A", "T_BILL", 0.010, 0.5),
            ("CCP_A", "UST_2Y", 0.020, 0.7),
            ("CCP_A", "UST_10Y", 0.040, 1.0),
            ("CCP_A", "CORP_BOND", 0.080, 1.5),

            ("PB_B_CASH_ONLY", "USD_CASH", 0.000, 0.1),

            ("CCP_C", "USD_CASH", 0.000, 0.2),
            ("CCP_C", "T_BILL", 0.015, 0.6),
            ("CCP_C", "UST_2Y", 0.030, 0.8),
            ("CCP_C", "UST_10Y", 0.050, 1.1),
        ],
        columns=[
            "counterparty",
            "asset",
            "haircut",
            "transfer_cost_bps",
        ],
    )

    # ------------------------------------------------------------------------
    # Repo routes
    # ------------------------------------------------------------------------

    repo_terms = pd.DataFrame(
        [
            (
                "D1_TBILL_OVERNIGHT",
                "REPO_D1",
                "T_BILL",
                0.02,
                20.0,
                50.0,
                1,
            ),
            (
                "D1_UST2_OVERNIGHT",
                "REPO_D1",
                "UST_2Y",
                0.03,
                28.0,
                45.0,
                1,
            ),
            (
                "D2_UST10_ONE_WEEK",
                "REPO_D2",
                "UST_10Y",
                0.05,
                45.0,
                40.0,
                7,
            ),
            (
                "D2_CORP_ONE_MONTH",
                "REPO_D2",
                "CORP_BOND",
                0.12,
                70.0,
                25.0,
                30,
            ),
        ],
        columns=[
            "route",
            "dealer",
            "asset",
            "haircut",
            "repo_cash_cost_bps",
            "maximum_collateral_usd_m",
            "tenor_days",
        ],
    )

    # ------------------------------------------------------------------------
    # Dealer-level repo limits
    # ------------------------------------------------------------------------

    repo_dealer_limits = pd.DataFrame(
        {
            "dealer": [
                "REPO_D1",
                "REPO_D2",
            ],
            "maximum_cash_proceeds_usd_m": [
                65.0,
                35.0,
            ],
        }
    )

    # ------------------------------------------------------------------------
    # Other cash funding sources
    # ------------------------------------------------------------------------

    cash_funding_sources = pd.DataFrame(
        [
            (
                "EUR_SURPLUS_VIA_FX_SWAP",
                "fx_swap",
                50.0,
                25.0,
                30,
            ),
            (
                "USD_PB_CASH_FACILITY",
                "pb_facility",
                85.0,
                60.0,
                1,
            ),
            (
                "USD_TERM_FACILITY",
                "term_facility",
                95.0,
                50.0,
                90,
            ),
        ],
        columns=[
            "source",
            "funding_type",
            "all_in_cost_bps",
            "maximum_cash_usd_m",
            "tenor_days",
        ],
    )

    # ------------------------------------------------------------------------
    # Liquidity policy
    # ------------------------------------------------------------------------

    cash_policy = CashPolicy(
        opening_cash_usd_m=45.0,
        settlement_cash_need_usd_m=20.0,
        minimum_liquidity_buffer_usd_m=25.0,
        minimum_stable_funding_usd_m=20.0,
        stable_tenor_days=30,
    )

    return (
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
    )


# ============================================================================
# PRINT INPUTS
# ============================================================================


def print_model_inputs(
    assets: pd.DataFrame,
    margin_requirements: pd.DataFrame,
    margin_terms: pd.DataFrame,
    repo_terms: pd.DataFrame,
    repo_dealer_limits: pd.DataFrame,
    cash_funding_sources: pd.DataFrame,
    cash_policy: CashPolicy,
) -> None:
    """Print the Treasury inputs in an interview-friendly format."""

    pd.set_option("display.width", 280)
    pd.set_option("display.max_columns", 40)
    pd.set_option(
        "display.float_format",
        lambda value: f"{value:,.4f}",
    )

    total_margin = float(
        margin_requirements["required_margin_usd_m"].sum()
    )

    total_security_inventory = float(
        assets.loc[
            assets["is_security"],
            "usable_inventory_usd_m",
        ].sum()
    )

    free_cash_available = (
        cash_policy.opening_cash_usd_m
        - cash_policy.settlement_cash_need_usd_m
        - cash_policy.minimum_liquidity_buffer_usd_m
    )

    # ========================================================================
    # 1. Problem
    # ========================================================================

    print("\n" + "=" * 110)
    print("1. PROBLEM TO SOLVE")
    print("=" * 110)

    print(
        """
We need to satisfy BOTH:

    A. Margin requirements at CCPs / prime brokers
    B. USD liquidity requirements

while minimizing annual economic cost.

The main trade-off is:

                         SECURITY INVENTORY
                                |
                --------------------------------
                |                              |
                v                              v
        POST FOR MARGIN                 PLEDGE INTO REPO
                |                              |
                v                              v
        Margin coverage                   Raise USD cash
                                                |
                                                v
                                   Settlements / liquidity

The same security cannot be used for both purposes at the same time.
"""
    )

    # ------------------------------------------------------------------------
    # Margin and inventory
    # ------------------------------------------------------------------------

    print("-" * 75)
    print("MARGIN AND SECURITY INVENTORY")
    print("-" * 75)

    print(
        f"Total margin requirement:               "
        f"${total_margin:,.2f}m"
    )

    print(
        f"Total available security inventory:     "
        f"${total_security_inventory:,.2f}m"
    )

    # ------------------------------------------------------------------------
    # Cash position
    # ------------------------------------------------------------------------

    print("\n" + "-" * 75)
    print("USD CASH POSITION")
    print("-" * 75)

    print(
        f"Opening cash                             "
        f"${cash_policy.opening_cash_usd_m:,.0f}m"
    )

    print(
        f"Needed for settlements                  "
        f"-${cash_policy.settlement_cash_need_usd_m:,.0f}m"
    )

    print(
        f"Must remain as liquidity buffer         "
        f"-${cash_policy.minimum_liquidity_buffer_usd_m:,.0f}m"
    )

    print(
        "                                         -----"
    )

    print(
        f"Free cash available                      "
        f"${free_cash_available:,.0f}m"
    )

    # ------------------------------------------------------------------------
    # Funding policy
    # ------------------------------------------------------------------------

    print("\n" + "-" * 75)
    print("FUNDING POLICY")
    print("-" * 75)

    print(
        f"Minimum stable funding "
        f"({cash_policy.stable_tenor_days}+ days):       "
        f"${cash_policy.minimum_stable_funding_usd_m:,.2f}m"
    )

    print(
        """
Interpretation:

Because free USD cash is $0m, any USD cash posted as margin generally
has to be replaced through repo, FX swaps, or another funding source.

The stable-funding requirement is different:
at least $20m of external funding must have a tenor of 30 days or more.
"""
    )

    # ========================================================================
    # 2. Assets
    # ========================================================================

    print("\n" + "=" * 110)
    print("2. AVAILABLE ASSETS")
    print("=" * 110)

    print(
        assets[
            [
                "asset",
                "is_security",
                "usable_inventory_usd_m",
                "opportunity_cost_bps",
            ]
        ].to_string(index=False)
    )

    print(
        """
Interpretation:

usable_inventory_usd_m
    Maximum market value of each security available for margin or repo.

opportunity_cost_bps
    Economic value lost when that asset becomes encumbered.

For example, if a Treasury is posted to a CCP, it can no longer be used
for repo or another collateral purpose.
"""
    )

    # ========================================================================
    # 3. Margin requirements
    # ========================================================================

    print("\n" + "=" * 110)
    print("3. COUNTERPARTY MARGIN REQUIREMENTS")
    print("=" * 110)

    print(
        margin_requirements.to_string(index=False)
    )

    # ========================================================================
    # 4. Collateral eligibility
    # ========================================================================

    print("\n" + "=" * 110)
    print("4. COLLATERAL ELIGIBILITY")
    print("=" * 110)

    eligibility = margin_terms.copy()
    eligibility["eligible"] = "YES"

    eligibility_matrix = (
        eligibility.pivot(
            index="asset",
            columns="counterparty",
            values="eligible",
        )
        .fillna("-")
        .reset_index()
    )

    print(
        eligibility_matrix.to_string(index=False)
    )

    print(
        """
A "-" means that asset is not eligible at that counterparty.

PB_B_CASH_ONLY accepts only USD cash.

That restriction matters because the optimizer must make sure enough
cash can be funded for PB_B while still preserving the liquidity buffer.
"""
    )

    # ========================================================================
    # 5. Margin economics
    # ========================================================================

    print("\n" + "=" * 110)
    print("5. MARGIN COLLATERAL ECONOMICS")
    print("=" * 110)

    margin_economics = margin_terms.merge(
        assets[
            [
                "asset",
                "opportunity_cost_bps",
            ]
        ],
        on="asset",
        how="left",
        validate="many_to_one",
    )

    margin_economics["raw_cost_bps"] = (
        margin_economics["opportunity_cost_bps"]
        + margin_economics["transfer_cost_bps"]
    )

    margin_economics["effective_margin_per_1"] = (
        1.0 - margin_economics["haircut"]
    )

    margin_economics["effective_cost_bps"] = (
        margin_economics["raw_cost_bps"]
        / margin_economics["effective_margin_per_1"]
    )

    print(
        margin_economics[
            [
                "counterparty",
                "asset",
                "haircut",
                "opportunity_cost_bps",
                "transfer_cost_bps",
                "raw_cost_bps",
                "effective_margin_per_1",
                "effective_cost_bps",
            ]
        ].to_string(index=False)
    )

    print(
        """
Margin collateral cost:

    raw cost
        = opportunity cost
        + transfer cost

Haircut determines usable margin:

    effective margin
        = collateral market value x (1 - haircut)

Therefore:

                          raw cost
    effective cost = ------------------
                        1 - haircut


Example: UST_10Y at CCP_A

    Opportunity cost = 25 bps
    Transfer cost    =  1 bps
                         ------
    Raw cost         = 26 bps

    Haircut = 4%

    Effective cost
        = 26 / 0.96
        = 27.08 bps
"""
    )

    # ========================================================================
    # 6. Repo routes
    # ========================================================================

    print("\n" + "=" * 110)
    print("6. REPO FUNDING OPTIONS")
    print("=" * 110)

    repo_economics = repo_terms.merge(
        assets[
            [
                "asset",
                "opportunity_cost_bps",
            ]
        ],
        on="asset",
        how="left",
        validate="many_to_one",
    )

    repo_economics["cash_per_1_collateral"] = (
        1.0 - repo_economics["haircut"]
    )

    repo_economics[
        "cost_per_collateral_bps"
    ] = (
        repo_economics["opportunity_cost_bps"]
        + repo_economics["repo_cash_cost_bps"]
        * repo_economics["cash_per_1_collateral"]
    )

    repo_economics[
        "effective_cash_cost_bps"
    ] = (
        repo_economics["cost_per_collateral_bps"]
        / repo_economics["cash_per_1_collateral"]
    )

    print(
        repo_economics[
            [
                "route",
                "dealer",
                "asset",
                "tenor_days",
                "haircut",
                "repo_cash_cost_bps",
                "opportunity_cost_bps",
                "maximum_collateral_usd_m",
                "cash_per_1_collateral",
                "effective_cash_cost_bps",
            ]
        ].to_string(index=False)
    )

    print(
        """
Repo economics:

    $1 security pledged
        -> haircut
        -> less than $1 cash received

Economic cost includes:

    1. opportunity cost of using the security
    2. repo financing cost on the cash raised

Therefore the optimizer should not simply choose the lowest quoted
repo rate.
"""
    )

    # ========================================================================
    # 7. Dealer capacity
    # ========================================================================

    print("\n" + "=" * 110)
    print("7. REPO DEALER CAPACITY")
    print("=" * 110)

    print(
        repo_dealer_limits.to_string(index=False)
    )

    print(
        """
These limits can represent:

    - credit capacity
    - concentration limits
    - dealer balance-sheet capacity
    - operational capacity
    - internal Treasury limits
"""
    )

    # ========================================================================
    # 8. Other funding
    # ========================================================================

    print("\n" + "=" * 110)
    print("8. OTHER USD FUNDING SOURCES")
    print("=" * 110)

    print(
        cash_funding_sources.to_string(index=False)
    )

    print(
        """
Example:

EUR_SURPLUS_VIA_FX_SWAP

means Treasury has surplus EUR liquidity and converts it into USD funding
through an FX swap.

The 50 bps is modeled as an all-in annualized USD funding cost.
"""
    )

    # ========================================================================
    # 9. LP formulation
    # ========================================================================

    print("\n" + "=" * 110)
    print("9. LINEAR PROGRAM FORMULATION")
    print("=" * 110)

    print(
        """
DECISION VARIABLES
------------------

M[c,a]
    Market value of asset a posted to counterparty c.

R[r]
    Market value of security pledged through repo route r.

F[f]
    USD cash raised using another funding source.


OBJECTIVE
---------

Minimize:

      collateral economic cost
    + repo funding economic cost
    + FX / facility funding cost


CONSTRAINT 1: MARGIN COVERAGE
-----------------------------

For each counterparty:

    SUM(
        collateral market value
        x
        (1 - haircut)
    )

        >= required margin


CONSTRAINT 2: SECURITY INVENTORY
--------------------------------

For each security:

    margin use
    +
    repo use

        <= available inventory


CONSTRAINT 3: USD CASH BALANCE
------------------------------

    opening cash
    + repo proceeds
    + other funding
    - cash posted as margin
    - settlement cash

        >= minimum liquidity buffer


CONSTRAINT 4: STABLE FUNDING
----------------------------

    funding with tenor >= 30 days

        >= minimum stable funding


CONSTRAINT 5: REPO DEALER CAPACITY
----------------------------------

    repo cash raised from dealer

        <= dealer limit
"""
    )

    print("\n" + "=" * 110)
    print("INPUT REVIEW COMPLETE -- RUNNING OPTIMIZER")
    print("=" * 110)


# ============================================================================
# BUILD DECISION VARIABLES
# ============================================================================


def build_decision_variables(
    assets: pd.DataFrame,
    margin_terms: pd.DataFrame,
    repo_terms: pd.DataFrame,
    cash_funding_sources: pd.DataFrame,
    *,
    allow_repo: bool = True,
    allow_fx_swap: bool = True,
) -> pd.DataFrame:
    """Create one row per LP decision variable."""

    asset_costs = assets[
        [
            "asset",
            "opportunity_cost_bps",
        ]
    ]

    # ------------------------------------------------------------------------
    # Margin variables
    # ------------------------------------------------------------------------

    margin_variables = margin_terms.merge(
        asset_costs,
        on="asset",
        how="left",
        validate="many_to_one",
    )

    margin_variables["variable"] = (
        "MARGIN::"
        + margin_variables["counterparty"]
        + "::"
        + margin_variables["asset"]
    )

    margin_variables["kind"] = "margin_collateral"

    margin_variables["destination"] = (
        margin_variables["counterparty"]
    )

    margin_variables["cash_use_per_unit"] = np.where(
        margin_variables["asset"].eq("USD_CASH"),
        1.0,
        0.0,
    )

    margin_variables["cash_proceeds_per_unit"] = 0.0

    margin_variables["effective_margin_per_unit"] = (
        1.0 - margin_variables["haircut"]
    )

    margin_variables["stable_cash_per_unit"] = 0.0

    margin_variables["upper_bound"] = np.inf

    margin_variables[
        "opportunity_cost_bps_per_decision_unit"
    ] = margin_variables[
        "opportunity_cost_bps"
    ]

    margin_variables[
        "transfer_cost_bps_per_decision_unit"
    ] = margin_variables[
        "transfer_cost_bps"
    ]

    margin_variables[
        "financing_cost_bps_per_decision_unit"
    ] = 0.0

    margin_variables[
        "cost_bps_per_decision_unit"
    ] = (
        margin_variables[
            "opportunity_cost_bps_per_decision_unit"
        ]
        + margin_variables[
            "transfer_cost_bps_per_decision_unit"
        ]
        + margin_variables[
            "financing_cost_bps_per_decision_unit"
        ]
    )

    margin_variables["dealer"] = None

    margin_variables["tenor_days"] = 0

    # ------------------------------------------------------------------------
    # Repo variables
    # ------------------------------------------------------------------------

    repo_variables = repo_terms.merge(
        asset_costs,
        on="asset",
        how="left",
        validate="many_to_one",
    )

    repo_variables["variable"] = (
        "REPO::"
        + repo_variables["route"]
    )

    repo_variables["kind"] = "repo_funding"

    repo_variables["destination"] = (
        repo_variables["route"]
    )

    repo_variables["cash_use_per_unit"] = 0.0

    repo_variables["cash_proceeds_per_unit"] = (
        1.0 - repo_variables["haircut"]
    )

    repo_variables["effective_margin_per_unit"] = 0.0

    repo_variables["stable_cash_per_unit"] = np.where(
        repo_variables["tenor_days"] >= 30,
        repo_variables["cash_proceeds_per_unit"],
        0.0,
    )

    repo_variables["upper_bound"] = np.where(
        allow_repo,
        repo_variables["maximum_collateral_usd_m"],
        0.0,
    )

    repo_variables[
        "opportunity_cost_bps_per_decision_unit"
    ] = repo_variables[
        "opportunity_cost_bps"
    ]

    repo_variables[
        "transfer_cost_bps_per_decision_unit"
    ] = 0.0

    # Repo rates apply to cash proceeds.  The LP decision is collateral
    # market value, so convert the financing component to the same unit.
    repo_variables[
        "financing_cost_bps_per_decision_unit"
    ] = (
        repo_variables["repo_cash_cost_bps"]
        * repo_variables["cash_proceeds_per_unit"]
    )

    repo_variables[
        "cost_bps_per_decision_unit"
    ] = (
        repo_variables[
            "opportunity_cost_bps_per_decision_unit"
        ]
        + repo_variables[
            "transfer_cost_bps_per_decision_unit"
        ]
        + repo_variables[
            "financing_cost_bps_per_decision_unit"
        ]
    )

    repo_variables["counterparty"] = None

    # ------------------------------------------------------------------------
    # Other cash funding variables
    # ------------------------------------------------------------------------

    cash_variables = cash_funding_sources.copy()

    cash_variables["variable"] = (
        "CASH_FUNDING::"
        + cash_variables["source"]
    )

    cash_variables["kind"] = "cash_funding"

    cash_variables["asset"] = None

    cash_variables["destination"] = (
        cash_variables["source"]
    )

    cash_variables["dealer"] = None

    cash_variables["haircut"] = 0.0

    cash_variables["cash_use_per_unit"] = 0.0

    cash_variables["cash_proceeds_per_unit"] = 1.0

    cash_variables["effective_margin_per_unit"] = 0.0

    cash_variables["stable_cash_per_unit"] = np.where(
        cash_variables["tenor_days"] >= 30,
        1.0,
        0.0,
    )

    cash_variables["upper_bound"] = (
        cash_variables["maximum_cash_usd_m"]
    )

    if not allow_fx_swap:
        cash_variables.loc[
            cash_variables["funding_type"].eq("fx_swap"),
            "upper_bound",
        ] = 0.0

    cash_variables[
        "opportunity_cost_bps_per_decision_unit"
    ] = 0.0

    cash_variables[
        "transfer_cost_bps_per_decision_unit"
    ] = 0.0

    cash_variables[
        "financing_cost_bps_per_decision_unit"
    ] = cash_variables[
        "all_in_cost_bps"
    ]

    cash_variables[
        "cost_bps_per_decision_unit"
    ] = (
        cash_variables[
            "opportunity_cost_bps_per_decision_unit"
        ]
        + cash_variables[
            "transfer_cost_bps_per_decision_unit"
        ]
        + cash_variables[
            "financing_cost_bps_per_decision_unit"
        ]
    )

    cash_variables["counterparty"] = None

    # ------------------------------------------------------------------------
    # Combine
    # ------------------------------------------------------------------------

    common_columns = [
        "variable",
        "kind",
        "asset",
        "destination",
        "counterparty",
        "dealer",
        "haircut",
        "tenor_days",
        "cash_use_per_unit",
        "cash_proceeds_per_unit",
        "effective_margin_per_unit",
        "stable_cash_per_unit",
        "opportunity_cost_bps_per_decision_unit",
        "transfer_cost_bps_per_decision_unit",
        "financing_cost_bps_per_decision_unit",
        "cost_bps_per_decision_unit",
        "upper_bound",
    ]

    variables = pd.concat(
        [
            margin_variables[common_columns],
            repo_variables[common_columns],
            cash_variables[common_columns],
        ],
        ignore_index=True,
    )

    variables["decision_index"] = np.arange(
        len(variables)
    )

    return variables


# ============================================================================
# PRINT OPTIMIZATION-ENGINE INPUTS
# ============================================================================


def print_optimization_engine_inputs(
    variables: pd.DataFrame,
    rows: list[np.ndarray],
    limits: list[float],
    names: list[str],
    objective: np.ndarray,
    bounds: list[tuple[float, float | None]],
) -> None:
    """Print the exact decision vector and inequality rows sent to linprog."""

    decision_units = {
        "margin_collateral":
            "collateral market value",
        "repo_funding":
            "collateral market value",
        "cash_funding":
            "cash raised",
    }

    engine_variables = variables.loc[
        :,
        [
            "decision_index",
            "variable",
            "kind",
            "opportunity_cost_bps_per_decision_unit",
            "transfer_cost_bps_per_decision_unit",
            "financing_cost_bps_per_decision_unit",
            "cost_bps_per_decision_unit",
        ],
    ].copy()

    engine_variables["decision_unit"] = (
        engine_variables["kind"].map(
            decision_units
        )
    )

    engine_variables["objective_c"] = objective

    engine_variables["lower_bound_usd_m"] = [
        lower
        for lower, _ in bounds
    ]

    engine_variables["upper_bound_usd_m"] = [
        "unbounded"
        if upper is None
        else f"{upper:,.4f}"
        for _, upper in bounds
    ]

    engine_variables = engine_variables.rename(
        columns={
            "decision_index": "index",
            "variable": "decision_variable",
            "kind": "decision_type",
            "opportunity_cost_bps_per_decision_unit":
                "opportunity_bps",
            "transfer_cost_bps_per_decision_unit":
                "transfer_bps",
            "financing_cost_bps_per_decision_unit":
                "financing_bps",
            "cost_bps_per_decision_unit":
                "total_objective_bps",
        }
    )[
        [
            "index",
            "decision_variable",
            "decision_type",
            "decision_unit",
            "opportunity_bps",
            "transfer_bps",
            "financing_bps",
            "total_objective_bps",
            "objective_c",
            "lower_bound_usd_m",
            "upper_bound_usd_m",
        ]
    ]

    clean_rows = [
        np.where(
            np.abs(row) <= 1e-12,
            0.0,
            row,
        )
        for row in rows
    ]

    coefficient_vectors = [
        np.array2string(
            row,
            separator=", ",
            formatter={
                "float_kind":
                    lambda value: f"{value:.4f}"
            },
            max_line_width=1_000,
        )
        for row in clean_rows
    ]

    engine_constraints = pd.DataFrame(
        {
            "row": np.arange(len(clean_rows)),
            "constraint": names,
            "coefficients_for_x0_to_x16":
                coefficient_vectors,
            "relation": "<=",
            "rhs_limit": limits,
        }
    )

    pd.set_option("display.width", 260)
    pd.set_option("display.max_columns", 40)
    pd.set_option(
        "display.float_format",
        lambda value: f"{value:,.6f}",
    )

    print("\n" + "#" * 110)
    print("OPTIMIZATION ENGINE INPUTS")
    print("#" * 110)

    print("\n" + "=" * 110)
    print("A. DECISION VECTOR x -- ALL 17 VARIABLES")
    print("=" * 110)
    print(
        "linprog minimizes c @ x.  "
        "objective_c = objective cost in bps / 10,000."
    )
    print(
        "All cost columns are annualized bps per decision unit; "
        "total_objective_bps = opportunity + transfer + financing."
    )
    print(
        "For repo, financing_bps equals the quoted repo rate multiplied "
        "by cash proceeds per collateral unit."
    )
    print(
        engine_variables.to_string(index=False)
    )

    print("\n" + "=" * 110)
    print("B. INEQUALITY ROWS -- ALL 11 CONSTRAINTS")
    print("=" * 110)
    print(
        f"A_ub shape: ({len(clean_rows)}, "
        f"{len(variables)})"
    )
    print(
        "Each row means: coefficient vector @ x <= rhs_limit.  "
        "Coefficient order is x[0] through x[16] from the table above."
    )
    print(
        engine_constraints.to_string(index=False)
    )

    print("\nDecoded nonzero terms:")

    for row_index, (
        constraint_name,
        row,
        limit,
    ) in enumerate(
        zip(
            names,
            clean_rows,
            limits,
        )
    ):
        terms = [
            f"{coefficient:+.4f}*x[{variable_index}]"
            for variable_index, coefficient in enumerate(row)
            if abs(coefficient) > 1e-12
        ]

        left_hand_side = " ".join(terms)

        if left_hand_side.startswith("+"):
            left_hand_side = left_hand_side[1:]

        print(
            f"row[{row_index:02d}] "
            f"{constraint_name}: "
            f"{left_hand_side} <= {limit:,.4f}"
        )

    print("\n" + "=" * 110)
    print("INPUT ASSEMBLY COMPLETE -- CALLING scipy.optimize.linprog")
    print("=" * 110)


# ============================================================================
# SOLVE LP
# ============================================================================


def solve_joint_problem(
    assets: pd.DataFrame,
    margin_requirements: pd.DataFrame,
    margin_terms: pd.DataFrame,
    repo_terms: pd.DataFrame,
    repo_dealer_limits: pd.DataFrame,
    cash_funding_sources: pd.DataFrame,
    cash_policy: CashPolicy,
    *,
    allow_repo: bool = True,
    allow_fx_swap: bool = True,
    print_engine_inputs: bool = False,
) -> JointSolution:
    """Solve the joint funding and collateral LP."""

    variables = build_decision_variables(
        assets,
        margin_terms,
        repo_terms,
        cash_funding_sources,
        allow_repo=allow_repo,
        allow_fx_swap=allow_fx_swap,
    )

    rows: list[np.ndarray] = []
    limits: list[float] = []
    names: list[str] = []
    categories: list[str] = []

    # ------------------------------------------------------------------------
    # 1. Security inventory
    # ------------------------------------------------------------------------

    security_assets = assets.loc[
        assets["is_security"]
    ]

    for asset_row in security_assets.itertuples(
        index=False
    ):
        uses_asset = (
            variables["asset"]
            .eq(asset_row.asset)
            .to_numpy(float)
        )

        rows.append(uses_asset)

        limits.append(
            float(asset_row.usable_inventory_usd_m)
        )

        names.append(
            f"inventory::{asset_row.asset}"
        )

        categories.append("inventory")

    # ------------------------------------------------------------------------
    # 2. Margin coverage
    # ------------------------------------------------------------------------

    for margin_row in margin_requirements.itertuples(
        index=False
    ):
        is_margin_for_counterparty = (
            variables["kind"].eq(
                "margin_collateral"
            )
            & variables["counterparty"].eq(
                margin_row.counterparty
            )
        ).to_numpy(float)

        coverage = (
            is_margin_for_counterparty
            * variables[
                "effective_margin_per_unit"
            ].to_numpy()
        )

        rows.append(-coverage)

        limits.append(
            -float(
                margin_row.required_margin_usd_m
            )
        )

        names.append(
            f"margin::{margin_row.counterparty}"
        )

        categories.append("margin")

    # ------------------------------------------------------------------------
    # 3. Cash balance
    # ------------------------------------------------------------------------

    cash_balance_row = (
        variables[
            "cash_use_per_unit"
        ].to_numpy()
        -
        variables[
            "cash_proceeds_per_unit"
        ].to_numpy()
    )

    rows.append(cash_balance_row)

    limits.append(
        cash_policy.opening_cash_usd_m
        - cash_policy.required_non_margin_cash_usd_m
    )

    names.append(
        "cash_balance::USD"
    )

    categories.append(
        "cash_balance"
    )

    # ------------------------------------------------------------------------
    # 4. Stable funding
    # ------------------------------------------------------------------------

    rows.append(
        -variables[
            "stable_cash_per_unit"
        ].to_numpy()
    )

    limits.append(
        -cash_policy.minimum_stable_funding_usd_m
    )

    names.append(
        "stable_funding::30_plus_days"
    )

    categories.append(
        "stable_funding"
    )

    # ------------------------------------------------------------------------
    # 5. Repo dealer capacity
    # ------------------------------------------------------------------------

    for dealer_row in repo_dealer_limits.itertuples(
        index=False
    ):
        is_dealer = (
            variables["dealer"]
            .eq(dealer_row.dealer)
            .to_numpy(float)
        )

        dealer_cash_proceeds = (
            is_dealer
            * variables[
                "cash_proceeds_per_unit"
            ].to_numpy()
        )

        rows.append(
            dealer_cash_proceeds
        )

        limits.append(
            float(
                dealer_row.maximum_cash_proceeds_usd_m
            )
        )

        names.append(
            f"repo_dealer_capacity::{dealer_row.dealer}"
        )

        categories.append(
            "repo_dealer_capacity"
        )

    # ------------------------------------------------------------------------
    # Objective
    # ------------------------------------------------------------------------

    objective = (
        variables[
            "cost_bps_per_decision_unit"
        ].to_numpy()
        / 10_000.0
    )

    bounds = [
        (
            0.0,
            None
            if np.isinf(upper)
            else float(upper),
        )
        for upper in variables["upper_bound"]
    ]

    if print_engine_inputs:
        print_optimization_engine_inputs(
            variables,
            rows,
            limits,
            names,
            objective,
            bounds,
        )

    # ------------------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------------------

    result = linprog(
        c=objective,
        A_ub=np.vstack(rows),
        b_ub=np.asarray(limits),
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            "Joint Treasury optimization failed: "
            f"{result.message}"
        )

    # ------------------------------------------------------------------------
    # Store solution
    # ------------------------------------------------------------------------

    variables = variables.copy()

    variables["decision_usd_m"] = result.x

    variables[
        "effective_margin_usd_m"
    ] = (
        variables["decision_usd_m"]
        * variables[
            "effective_margin_per_unit"
        ]
    )

    variables[
        "cash_raised_usd_m"
    ] = (
        variables["decision_usd_m"]
        * variables[
            "cash_proceeds_per_unit"
        ]
    )

    variables[
        "annual_cost_usd_m"
    ] = (
        result.x
        * objective
    )

    variables["effective_cost_bps"] = np.where(
        variables["kind"].eq(
            "margin_collateral"
        ),
        (
            variables[
                "cost_bps_per_decision_unit"
            ]
            /
            variables[
                "effective_margin_per_unit"
            ]
        ),
        (
            variables[
                "cost_bps_per_decision_unit"
            ]
            /
            variables[
                "cash_proceeds_per_unit"
            ].replace(0.0, np.nan)
        ),
    )

    constraints = pd.DataFrame(
        {
            "constraint": names,
            "category": categories,
            "limit": limits,
            "slack": result.ineqlin.residual,
            "solver_marginal": result.ineqlin.marginals,
        }
    )

    return JointSolution(
        result=result,
        variables=variables,
        constraints=constraints,
    )


# ============================================================================
# SUMMARIZE RESULTS
# ============================================================================


def summarize_solution(
    solution: JointSolution,
    assets: pd.DataFrame,
    margin_requirements: pd.DataFrame,
    cash_policy: CashPolicy,
) -> dict[str, float]:
    """Print economically meaningful outputs from the LP."""

    variables = solution.variables

    active = variables.loc[
        variables["decision_usd_m"] > 1e-8
    ].copy()

    # ------------------------------------------------------------------------
    # Complete decision vector, including zero-valued decisions
    # ------------------------------------------------------------------------

    all_decisions = variables.loc[
        :,
        [
            "decision_index",
            "variable",
            "kind",
            "decision_usd_m",
            "opportunity_cost_bps_per_decision_unit",
            "transfer_cost_bps_per_decision_unit",
            "financing_cost_bps_per_decision_unit",
            "cost_bps_per_decision_unit",
        ],
    ].copy()

    # Replace tiny numerical solver values with an exact zero for display.
    zero_decision_mask = (
        all_decisions["decision_usd_m"].abs() <= 1e-8
    )

    all_decisions.loc[
        zero_decision_mask,
        "decision_usd_m",
    ] = 0.0

    # With decision values in USD millions and costs in annualized bps:
    # annual cost in USD = decision_usd_m * component_bps * 100.
    cost_components = {
        "opportunity_cost_bps_per_decision_unit":
            "opportunity_cost_usd",
        "transfer_cost_bps_per_decision_unit":
            "transfer_cost_usd",
        "financing_cost_bps_per_decision_unit":
            "financing_cost_usd",
    }

    for bps_column, usd_column in cost_components.items():
        all_decisions[usd_column] = (
            all_decisions["decision_usd_m"]
            * all_decisions[bps_column]
            * 100.0
        )

    all_decisions["total_annual_cost_usd"] = (
        all_decisions[
            list(cost_components.values())
        ].sum(axis=1)
    )

    all_decisions.loc[
        zero_decision_mask,
        [
            *cost_components.values(),
            "total_annual_cost_usd",
        ],
    ] = 0.0

    all_decisions["decision_unit"] = (
        all_decisions["kind"].map(
            {
                "margin_collateral":
                    "collateral market value",
                "repo_funding":
                    "collateral market value",
                "cash_funding":
                    "cash raised",
            }
        )
    )

    all_decisions["status"] = np.where(
        all_decisions["decision_usd_m"] > 1e-8,
        "ACTIVE",
        "ZERO",
    )

    all_decisions = all_decisions.rename(
        columns={
            "decision_index": "index",
            "variable": "decision_variable",
            "kind": "decision_type",
            "decision_usd_m": "optimal_value_usd_m",
            "cost_bps_per_decision_unit":
                "total_objective_bps",
        }
    )[
        [
            "index",
            "decision_variable",
            "decision_type",
            "decision_unit",
            "optimal_value_usd_m",
            "total_objective_bps",
            "opportunity_cost_usd",
            "transfer_cost_usd",
            "financing_cost_usd",
            "total_annual_cost_usd",
            "status",
        ]
    ]

    total_opportunity_cost_usd = float(
        all_decisions[
            "opportunity_cost_usd"
        ].sum()
    )

    total_transfer_cost_usd = float(
        all_decisions[
            "transfer_cost_usd"
        ].sum()
    )

    total_financing_cost_usd = float(
        all_decisions[
            "financing_cost_usd"
        ].sum()
    )

    total_all_decision_cost_usd = float(
        all_decisions[
            "total_annual_cost_usd"
        ].sum()
    )

    solver_objective_usd = float(
        solution.result.fun
        * 1_000_000.0
    )

    # ------------------------------------------------------------------------
    # Margin plan
    # ------------------------------------------------------------------------

    margin_plan = active.loc[
        active["kind"].eq(
            "margin_collateral"
        ),
        [
            "counterparty",
            "asset",
            "haircut",
            "decision_usd_m",
            "effective_margin_usd_m",
            "effective_cost_bps",
            "annual_cost_usd_m",
        ],
    ].rename(
        columns={
            "decision_usd_m":
                "posted_market_value_usd_m"
        }
    )

    # ------------------------------------------------------------------------
    # Funding plan
    # ------------------------------------------------------------------------

    funding_plan = active.loc[
        active["kind"].isin(
            [
                "repo_funding",
                "cash_funding",
            ]
        ),
        [
            "kind",
            "destination",
            "asset",
            "tenor_days",
            "decision_usd_m",
            "cash_raised_usd_m",
            "effective_cost_bps",
            "annual_cost_usd_m",
        ],
    ].rename(
        columns={
            "decision_usd_m":
                "decision_or_collateral_usd_m"
        }
    )

    # ------------------------------------------------------------------------
    # Margin coverage
    # ------------------------------------------------------------------------

    margin_coverage = (
        margin_plan.groupby(
            "counterparty",
            as_index=False,
        )[
            "effective_margin_usd_m"
        ]
        .sum()
        .merge(
            margin_requirements,
            on="counterparty",
            how="right",
            validate="one_to_one",
        )
    )

    margin_coverage[
        "effective_margin_usd_m"
    ] = margin_coverage[
        "effective_margin_usd_m"
    ].fillna(0.0)

    margin_coverage[
        "excess_margin_usd_m"
    ] = (
        margin_coverage[
            "effective_margin_usd_m"
        ]
        -
        margin_coverage[
            "required_margin_usd_m"
        ]
    )

    # ------------------------------------------------------------------------
    # Security usage
    # ------------------------------------------------------------------------

    security_usage = (
        active.loc[
            active["asset"].notna()
            & active["asset"].ne("USD_CASH")
        ]
        .groupby(
            [
                "asset",
                "kind",
            ],
            as_index=False,
        )["decision_usd_m"]
        .sum()
        .pivot(
            index="asset",
            columns="kind",
            values="decision_usd_m",
        )
        .fillna(0.0)
        .reset_index()
        .merge(
            assets.loc[
                assets["is_security"],
                [
                    "asset",
                    "usable_inventory_usd_m",
                ],
            ],
            on="asset",
            how="right",
            validate="one_to_one",
        )
    )

    for column in [
        "margin_collateral",
        "repo_funding",
    ]:
        if column not in security_usage:
            security_usage[column] = 0.0

    security_usage[
        [
            "margin_collateral",
            "repo_funding",
        ]
    ] = security_usage[
        [
            "margin_collateral",
            "repo_funding",
        ]
    ].fillna(0.0)

    security_usage[
        "total_used_usd_m"
    ] = (
        security_usage[
            "margin_collateral"
        ]
        +
        security_usage[
            "repo_funding"
        ]
    )

    security_usage[
        "unused_usd_m"
    ] = (
        security_usage[
            "usable_inventory_usd_m"
        ]
        -
        security_usage[
            "total_used_usd_m"
        ]
    )

    # ------------------------------------------------------------------------
    # Cash
    # ------------------------------------------------------------------------

    cash_margin_posted = float(
        active.loc[
            active["kind"].eq(
                "margin_collateral"
            )
            & active["asset"].eq(
                "USD_CASH"
            ),
            "decision_usd_m",
        ].sum()
    )

    repo_cash = float(
        active.loc[
            active["kind"].eq(
                "repo_funding"
            ),
            "cash_raised_usd_m",
        ].sum()
    )

    other_funding_cash = float(
        active.loc[
            active["kind"].eq(
                "cash_funding"
            ),
            "cash_raised_usd_m",
        ].sum()
    )

    ending_cash = (
        cash_policy.opening_cash_usd_m
        + repo_cash
        + other_funding_cash
        - cash_margin_posted
        - cash_policy.settlement_cash_need_usd_m
    )

    # ------------------------------------------------------------------------
    # Costs
    # ------------------------------------------------------------------------

    collateral_cost = float(
        active.loc[
            active["kind"].eq(
                "margin_collateral"
            ),
            "annual_cost_usd_m",
        ].sum()
    )

    funding_cost = float(
        active.loc[
            active["kind"].isin(
                [
                    "repo_funding",
                    "cash_funding",
                ]
            ),
            "annual_cost_usd_m",
        ].sum()
    )

    total_cash_raised = (
        repo_cash
        + other_funding_cash
    )

    if total_cash_raised > 0:
        weighted_funding_cost_bps = (
            funding_cost
            / total_cash_raised
            * 10_000.0
        )
    else:
        weighted_funding_cost_bps = np.nan

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 30)
    pd.set_option(
        "display.float_format",
        lambda value: f"{value:,.4f}",
    )

    # ------------------------------------------------------------------------
    # Complete optimal decision vector
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("OPTIMAL SOLUTION -- ALL 17 DECISION VARIABLES")
    print("USD millions; zero-valued decisions are intentionally included")
    print(
        "Each annual USD cost component = optimal value in USD millions "
        "x component cost in bps x 100"
    )
    print(
        "A zero component means that cost type is not separately modeled "
        "for that decision."
    )
    print("=" * 110)

    print(
        all_decisions.to_string(index=False)
    )

    print("-" * 110)
    print(
        "TOTAL OPPORTUNITY COST:                         "
        f"${total_opportunity_cost_usd:,.2f}"
    )
    print(
        "TOTAL TRANSFER COST:                            "
        f"${total_transfer_cost_usd:,.2f}"
    )
    print(
        "TOTAL FINANCING COST:                           "
        f"${total_financing_cost_usd:,.2f}"
    )
    print(
        "SUM OF ANNUAL COSTS ACROSS ALL 17 DECISIONS: "
        f"${total_all_decision_cost_usd:,.2f}"
    )
    print(
        "SOLVER OBJECTIVE VALUE:                         "
        f"${solver_objective_usd:,.2f}"
    )
    print(
        "RECONCILIATION DIFFERENCE:                      "
        f"${total_all_decision_cost_usd - solver_objective_usd:,.8f}"
    )

    # ------------------------------------------------------------------------
    # Optimal margin collateral
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("OPTIMAL MARGIN COLLATERAL")
    print("USD millions")
    print("=" * 110)

    print(
        margin_plan.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Margin coverage
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("MARGIN COVERAGE CHECK")
    print("=" * 110)

    print(
        margin_coverage.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Funding plan
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("OPTIMAL FUNDING PLAN")
    print("USD millions")
    print("=" * 110)

    print(
        funding_plan.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Inventory
    # ------------------------------------------------------------------------

    print("\n" + "=" * 110)
    print("SHARED SECURITY INVENTORY CHECK")
    print("=" * 110)

    print(
        security_usage[
            [
                "asset",
                "margin_collateral",
                "repo_funding",
                "total_used_usd_m",
                "usable_inventory_usd_m",
                "unused_usd_m",
            ]
        ].to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Cash reconciliation
    # ------------------------------------------------------------------------

    cash_reconciliation = pd.DataFrame(
        {
            "cash_item": [
                "Opening USD cash",
                "Repo cash raised",
                "Other / FX cash raised",
                "Cash posted for margin",
                "Settlement cash need",
                "Ending USD cash",
                "Minimum liquidity buffer",
            ],
            "usd_m": [
                cash_policy.opening_cash_usd_m,
                repo_cash,
                other_funding_cash,
                -cash_margin_posted,
                -cash_policy.settlement_cash_need_usd_m,
                ending_cash,
                cash_policy.minimum_liquidity_buffer_usd_m,
            ],
        }
    )

    print("\n" + "=" * 110)
    print("USD CASH RECONCILIATION")
    print("=" * 110)

    print(
        cash_reconciliation.to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Cost summary
    # ------------------------------------------------------------------------

    cost_summary = pd.DataFrame(
        {
            "cost_component": [
                "Margin-collateral economic cost",
                "Funding economic cost",
                "Total annual economic cost",
            ],
            "annual_cost_usd_m": [
                collateral_cost,
                funding_cost,
                solution.result.fun,
            ],
            "annual_cost_usd": [
                collateral_cost * 1_000_000,
                funding_cost * 1_000_000,
                solution.result.fun * 1_000_000,
            ],
        }
    )

    print("\n" + "=" * 110)
    print("COST SUMMARY")
    print("=" * 110)

    print(
        cost_summary.to_string(index=False)
    )

    print(
        f"\nWeighted effective cost of cash raised: "
        f"{weighted_funding_cost_bps:,.2f} bps"
    )

    # ------------------------------------------------------------------------
    # Marginal costs
    # ------------------------------------------------------------------------

    useful_duals = (
        solution.constraints.loc[
            solution.constraints[
                "category"
            ].isin(
                [
                    "margin",
                    "cash_balance",
                    "stable_funding",
                ]
            )
        ]
        .copy()
    )

    useful_duals[
        "marginal_cost_bps"
    ] = (
        -useful_duals[
            "solver_marginal"
        ]
        * 10_000.0
    )

    print("\n" + "=" * 110)
    print("MARGINAL COST OF USD 1mm MORE REQUIREMENT")
    print("=" * 110)

    print(
        useful_duals[
            [
                "constraint",
                "slack",
                "marginal_cost_bps",
            ]
        ].to_string(index=False)
    )

    # ------------------------------------------------------------------------
    # Scarce capacity
    # ------------------------------------------------------------------------

    scarce_capacity_duals = (
        solution.constraints.loc[
            solution.constraints[
                "category"
            ].isin(
                [
                    "inventory",
                    "repo_dealer_capacity",
                ]
            )
        ]
        .copy()
    )

    scarce_capacity_duals[
        "value_of_one_more_usd_1mm_bps"
    ] = (
        -scarce_capacity_duals[
            "solver_marginal"
        ]
        * 10_000.0
    )

    print("\n" + "=" * 110)
    print("VALUE OF USD 1mm MORE INVENTORY OR DEALER CAPACITY")
    print("=" * 110)

    print(
        scarce_capacity_duals[
            [
                "constraint",
                "slack",
                "value_of_one_more_usd_1mm_bps",
            ]
        ].to_string(index=False)
    )

    return {
        "total_annual_cost_usd_m":
            float(solution.result.fun),

        "collateral_cost_usd_m":
            collateral_cost,

        "funding_cost_usd_m":
            funding_cost,

        "total_cash_raised_usd_m":
            total_cash_raised,

        "weighted_funding_cost_bps":
            weighted_funding_cost_bps,

        "ending_cash_usd_m":
            ending_cash,
    }


# ============================================================================
# MAIN
# ============================================================================


def main() -> None:

    (
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
    ) = make_synthetic_inputs()

    print("\n" + "#" * 110)
    print("JOINT FUNDING AND COLLATERAL OPTIMIZATION")
    print("#" * 110)

    print(
        """
All data are synthetic.
All monetary amounts are USD millions.
All rates and costs are annualized basis points.

Purpose:

    Illustrate how collateral allocation and funding efficiency
    can be optimized jointly.
"""
    )

    # ------------------------------------------------------------------------
    # Print inputs
    # ------------------------------------------------------------------------

    print_model_inputs(
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
    )

    # ------------------------------------------------------------------------
    # Solve joint optimum
    # ------------------------------------------------------------------------

    joint_solution = solve_joint_problem(
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
        print_engine_inputs=True,
    )

    print("\n" + "#" * 110)
    print("OPTIMIZATION SUCCESSFUL")
    print("#" * 110)

    print(
        f"Solver status:      "
        f"{joint_solution.result.message}"
    )

    print(
        f"Solver iterations:  "
        f"{joint_solution.result.nit}"
    )

    print(
        f"Decision variables: "
        f"{len(joint_solution.variables)}"
    )

    print(
        f"LP constraints:     "
        f"{len(joint_solution.constraints)}"
    )

    joint_summary = summarize_solution(
        joint_solution,
        assets,
        margin_requirements,
        cash_policy,
    )

    # ------------------------------------------------------------------------
    # Counterfactual: no repo
    # ------------------------------------------------------------------------

    print("\n" + "#" * 110)
    print("RESEARCH COUNTERFACTUAL: WHAT IF REPO WERE UNAVAILABLE?")
    print("#" * 110)

    print(
        """
We re-optimize the entire Treasury problem with repo disabled.

This lets collateral allocation and other funding sources adjust
optimally.

The increase in minimum economic cost represents the modeled value
of having repo available.
"""
    )

    no_repo_solution = solve_joint_problem(
        assets,
        margin_requirements,
        margin_terms,
        repo_terms,
        repo_dealer_limits,
        cash_funding_sources,
        cash_policy,
        allow_repo=False,
    )

    no_repo_total_cost = float(
        no_repo_solution.result.fun
    )

    annual_savings = (
        no_repo_total_cost
        - joint_summary[
            "total_annual_cost_usd_m"
        ]
    )

    comparison = pd.DataFrame(
        {
            "scenario": [
                "Joint optimum",
                "Counterfactual: repo unavailable",
            ],
            "annual_economic_cost_usd_m": [
                joint_summary[
                    "total_annual_cost_usd_m"
                ],
                no_repo_total_cost,
            ],
            "annual_economic_cost_usd": [
                joint_summary[
                    "total_annual_cost_usd_m"
                ]
                * 1_000_000,

                no_repo_total_cost
                * 1_000_000,
            ],
        }
    )

    print("\nRESEARCH COUNTERFACTUAL")

    print(
        comparison.to_string(index=False)
    )

    print(
        f"\nEstimated annual value of repo availability: "
        f"${annual_savings * 1_000_000:,.0f}"
    )

    # ------------------------------------------------------------------------
    # Final interpretation
    # ------------------------------------------------------------------------

    print("\n" + "#" * 110)
    print("FINAL ECONOMIC INTERPRETATION")
    print("#" * 110)

    print(
        f"""
Optimal annual economic cost:
    ${joint_summary["total_annual_cost_usd_m"] * 1_000_000:,.0f}

Collateral allocation cost:
    ${joint_summary["collateral_cost_usd_m"] * 1_000_000:,.0f}

Funding cost:
    ${joint_summary["funding_cost_usd_m"] * 1_000_000:,.0f}

Total external cash raised:
    ${joint_summary["total_cash_raised_usd_m"]:,.2f}m

Weighted effective funding cost:
    {joint_summary["weighted_funding_cost_bps"]:,.2f} bps

Ending USD liquidity:
    ${joint_summary["ending_cash_usd_m"]:,.2f}m


CORE TREASURY QUESTION
----------------------

For every scarce security, the optimizer effectively asks:

    Where does this asset create the most economic value?

Possible uses:

    1. Post it as margin collateral
    2. Pledge it into repo and raise USD cash
    3. Leave it unencumbered


The key linking constraint is:

    MARGIN USE
    +
    REPO USE

        <= AVAILABLE SECURITY INVENTORY


That is what makes this a joint funding-efficiency and
collateral-allocation optimization problem.
"""
    )


if __name__ == "__main__":
    main()


##############################################################################################################
JOINT FUNDING AND COLLATERAL OPTIMIZATION
##############################################################################################################

All data are synthetic.
All monetary amounts are USD millions.
All rates and costs are annualized basis points.

Purpose:

    Illustrate how collateral allocation and funding efficiency
    can be optimized jointly.


1. PROBLEM TO SOLVE

We need to satisfy BOTH:

    A. Margin requirements at CCPs / prime brokers
    B. USD liquidity requirements

while minimizing annual economic cost.

The main trade-off is:

                         SECURITY INVENTORY
                                |
                --------------------------------
                |                              |
                v                              v
        POST FOR MARGIN                 PLEDGE INTO REPO
                |                   